<a href="https://colab.research.google.com/github/Nahmadid/SpectralBias/blob/main/gif_spctral_bias_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import jax
import jax.numpy as jnp
import optax
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

# === 1. Target function with multiple frequencies ===
def target_function(t):
    return jnp.sin(2 * jnp.pi * 0.01 * t) + 0.5 * jnp.sin(2 * jnp.pi * 0.05 * t) + 0.2 * jnp.sin(2 * jnp.pi * 0.1 * t)

# === 2. Chebyshev recursive basis ===
def chebyshev_recursive(x, degree):
    T_n_minus_2 = x * 0 + 1
    T_n_minus_1 = x
    result = [T_n_minus_2, T_n_minus_1]
    for n in range(2, degree + 1):
        T_n = 2 * x * T_n_minus_1 - T_n_minus_2
        result.append(T_n)
        T_n_minus_2, T_n_minus_1 = T_n_minus_1, T_n
    return jnp.stack(result, axis=-1)

# === 3. Gated cPIKAN initializer ===
def init_params_kan2(layers, degree, key=jax.random.PRNGKey(123)):
    keys = jax.random.split(key, len(layers))
    params = []
    for i in range(len(layers) - 2):
        W = jax.random.normal(keys[i], shape=(layers[i], layers[i+1], degree + 1)) / jnp.sqrt(layers[i] * (degree + 1))
        g = jax.random.normal(keys[i], shape=(layers[i+1],))
        params.append({'W': W, 'g': g})
    W = jax.random.normal(keys[-1], shape=(layers[-2], layers[-1])) / jnp.sqrt(layers[-2])
    B = jax.random.normal(keys[-1], shape=(layers[-1],))
    params.append({'W': W, 'B': B})
    return params

# === 4. Forward pass ===
def fwd(params, t, activation=jax.nn.tanh):
    t = 0.01 * t
    X = t.reshape((-1, 1))
    *hidden, last = params
    for layer in hidden:
        W = layer['W']
        g = layer['g']
        degree = W.shape[-1] - 1
        X_stack = chebyshev_recursive(X, degree)
        X = jnp.einsum("bid,iod->bo", X_stack, W)
        X = g * X
        X = activation(X)
    return X @ last['W'] + last['B']

# === 5. Loss ===
def mse_loss(params, t, y_true):
    y_pred = fwd(params, t)
    return jnp.mean((y_pred[:, 0] - y_true) ** 2)

# === 6. FFT Spectrum ===
def compute_fourier_spectrum(signal, dt):
    n = len(signal)
    freqs = np.fft.fftfreq(n, d=dt)
    fft_vals = np.fft.fft(signal)
    magnitude = np.abs(fft_vals)
    return freqs[:n // 2], magnitude[:n // 2]

# === 7. Setup ===
layers = [1, 64, 64, 1]
degree = 5
lr = 1e-4
epochs = 40000
log_epochs = list(range(0, epochs + 1, 1000))

t = jnp.linspace(0, 300, 301)
y_true = target_function(t)

params = init_params_kan2(layers, degree)
optimizer = optax.adam(lr)
opt_state = optimizer.init(params)

# === 8. Training step ===
@jax.jit
def train_step(params, opt_state, t, y_true):
    loss, grads = jax.value_and_grad(mse_loss)(params, t, y_true)
    updates, opt_state = optimizer.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss

# === 9. Training loop ===
predictions = {}
loss_history = []

for epoch in range(epochs + 1):
    params, opt_state, loss = train_step(params, opt_state, t, y_true)
    loss_history.append(loss)
    if epoch in log_epochs:
        predictions[epoch] = np.array(fwd(params, t)[:, 0])

# === 10. Animation: Signal vs. Prediction + Spectrum ===
plt.switch_backend("Agg")
dt = float(t[1] - t[0])
log_epochs = sorted(predictions.keys())

freqs = np.fft.fftfreq(len(t), d=dt)
mask = freqs >= 0
freqs_pos = freqs[mask]
fft_true = np.fft.fft(np.array(y_true))
amp_true = np.abs(fft_true)[mask]

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
plt.tight_layout()

def animate(i):
    epoch = log_epochs[i]
    y_pred = predictions[epoch]
    amp_pred = np.abs(np.fft.fft(y_pred))[mask]

    axs[0].clear()
    axs[0].plot(t, y_true, label='True', color='black')
    axs[0].plot(t, y_pred, '--', label=f'Pred', color='red')
    axs[0].set_title(f"Signal Prediction at Epoch {epoch}")
    axs[0].set_xlabel("Time")
    axs[0].set_ylabel("Signal")
    axs[0].legend()
    axs[0].grid(True)

    axs[1].clear()
    axs[1].plot(freqs_pos, amp_true, label='True Spectrum', color='black')
    axs[1].plot(freqs_pos, amp_pred, '--', label='Predicted Spectrum', color='red')
    axs[1].set_title(f"Frequency Spectrum at Epoch {epoch}")
    axs[1].set_xlabel("Frequency (Hz)")
    axs[1].set_ylabel("Amplitude")
    axs[1].set_xlim(0, 0.2)
    axs[1].set_yscale("log")
    axs[1].grid(True, which="both", ls="--", alpha=0.5)
    axs[1].legend()

anim = FuncAnimation(fig, animate, frames=len(log_epochs), interval=500)

# Save animation
anim.save("training_evolution_gkan.gif", writer=PillowWriter(fps=2))
anim.save("training_evolution_gkan.mp4", fps=2, extra_args=['-vcodec', 'libx264'])

print("Saved animation: training_evolution.gif, training_evolution.mp4")


Saved animation: training_evolution.gif, training_evolution.mp4


In [ ]:

anim.save("gkan_disc.gif", writer=PillowWriter(fps=2))

In [ ]:
import jax
import jax.numpy as jnp
import optax
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

# === 1. Target function with multiple frequencies ===
def target_function(t):
    return jnp.sin(2 * jnp.pi * 0.01 * t) + 0.5 * jnp.sin(2 * jnp.pi * 0.05 * t) + 0.2 * jnp.sin(2 * jnp.pi * 0.1 * t)

# === 2. Chebyshev recursive basis ===
def chebyshev_recursive(x, degree):
    T_n_minus_2 = x * 0 + 1
    T_n_minus_1 = x
    result = [T_n_minus_2, T_n_minus_1]
    for n in range(2, degree + 1):
        T_n = 2 * x * T_n_minus_1 - T_n_minus_2
        result.append(T_n)
        T_n_minus_2, T_n_minus_1 = T_n_minus_1, T_n
    return jnp.stack(result, axis=-1)

# === 3. Gated cPIKAN initializer ===
def init_params_kan2(layers, degree, key=jax.random.PRNGKey(123)):
    keys = jax.random.split(key, len(layers))
    params = []
    for i in range(len(layers) - 2):
        W = jax.random.normal(keys[i], shape=(layers[i], layers[i+1], degree + 1)) / jnp.sqrt(layers[i] * (degree + 1))
        g = jax.random.normal(keys[i], shape=(layers[i+1],))
        params.append({'W': W, 'g': g})
    W = jax.random.normal(keys[-1], shape=(layers[-2], layers[-1])) / jnp.sqrt(layers[-2])
    B = jax.random.normal(keys[-1], shape=(layers[-1],))
    params.append({'W': W, 'B': B})
    return params

# === 4. Forward pass ===
def fwd(params, t, activation=jax.nn.tanh):
    t = 0.01 * t
    X = t.reshape((-1, 1))
    *hidden, last = params
    for layer in hidden:
        W = layer['W']
        g = layer['g']
        degree = W.shape[-1] - 1
        X_stack = chebyshev_recursive(X, degree)
        X = activation(X)
        X = jnp.einsum("bid,iod->bo", X_stack, W)
        # X = g * X
        X = activation(X)
    return X @ last['W'] + last['B']

# === 5. Loss ===
def mse_loss(params, t, y_true):
    y_pred = fwd(params, t)
    return jnp.mean((y_pred[:, 0] - y_true) ** 2)

# === 6. FFT Spectrum ===
def compute_fourier_spectrum(signal, dt):
    n = len(signal)
    freqs = np.fft.fftfreq(n, d=dt)
    fft_vals = np.fft.fft(signal)
    magnitude = np.abs(fft_vals)
    return freqs[:n // 2], magnitude[:n // 2]

# === 7. Setup ===
layers = [1, 64, 64, 1]
degree = 5
lr = 1e-4
epochs = 40000
log_epochs = list(range(0, epochs + 1, 1000))

t = jnp.linspace(0, 300, 301)
y_true = target_function(t)

params = init_params_kan2(layers, degree)
optimizer = optax.adam(lr)
opt_state = optimizer.init(params)

# === 8. Training step ===
@jax.jit
def train_step(params, opt_state, t, y_true):
    loss, grads = jax.value_and_grad(mse_loss)(params, t, y_true)
    updates, opt_state = optimizer.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss

# === 9. Training loop ===
predictions = {}
loss_history = []

for epoch in range(epochs + 1):
    params, opt_state, loss = train_step(params, opt_state, t, y_true)
    loss_history.append(loss)
    if epoch in log_epochs:
        predictions[epoch] = np.array(fwd(params, t)[:, 0])

# === 10. Animation: Signal vs. Prediction + Spectrum ===
plt.switch_backend("Agg")
dt = float(t[1] - t[0])
log_epochs = sorted(predictions.keys())

freqs = np.fft.fftfreq(len(t), d=dt)
mask = freqs >= 0
freqs_pos = freqs[mask]
fft_true = np.fft.fft(np.array(y_true))
amp_true = np.abs(fft_true)[mask]

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
plt.tight_layout()

def animate(i):
    epoch = log_epochs[i]
    y_pred = predictions[epoch]
    amp_pred = np.abs(np.fft.fft(y_pred))[mask]

    axs[0].clear()
    axs[0].plot(t, y_true, label='True', color='black')
    axs[0].plot(t, y_pred, '--', label=f'Pred', color='red')
    axs[0].set_title(f"Signal Prediction at Epoch {epoch}")
    axs[0].set_xlabel("Time")
    axs[0].set_ylabel("Signal")
    axs[0].legend()
    axs[0].grid(True)

    axs[1].clear()
    axs[1].plot(freqs_pos, amp_true, label='True Spectrum', color='black')
    axs[1].plot(freqs_pos, amp_pred, '--', label='Predicted Spectrum', color='red')
    axs[1].set_title(f"Frequency Spectrum at Epoch {epoch}")
    axs[1].set_xlabel("Frequency (Hz)")
    axs[1].set_ylabel("Amplitude")
    axs[1].set_xlim(0, 0.2)
    axs[1].set_yscale("log")
    axs[1].grid(True, which="both", ls="--", alpha=0.5)
    axs[1].legend()

anim = FuncAnimation(fig, animate, frames=len(log_epochs), interval=500)

# Save animation
anim.save("training_evolution_tanhcPIKAN.gif", writer=PillowWriter(fps=2))
anim.save("training_evolution_tanhcPIKAN.mp4", fps=2, extra_args=['-vcodec', 'libx264'])

print("Saved animation: training_evolution.gif, training_evolution.mp4")


Saved animation: training_evolution.gif, training_evolution.mp4


#siren

#other

In [ ]:
import jax
import jax.numpy as jnp
import optax
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

# === 1. Target function with multiple frequencies ===
def target_function(t):
    return jnp.sin(2 * jnp.pi * 0.01 * t) + 0.5 * jnp.sin(2 * jnp.pi * 0.05 * t) + 0.2 * jnp.sin(2 * jnp.pi * 0.1 * t)

# === 2. Chebyshev recursive basis ===
def chebyshev_recursive(x, degree):
    T_n_minus_2 = x * 0 + 1
    T_n_minus_1 = x
    result = [T_n_minus_2, T_n_minus_1]
    for n in range(2, degree + 1):
        T_n = 2 * x * T_n_minus_1 - T_n_minus_2
        result.append(T_n)
        T_n_minus_2, T_n_minus_1 = T_n_minus_1, T_n
    return jnp.stack(result, axis=-1)

# === 3. Gated cPIKAN initializer ===
def init_params_kan2(layers, degree, key=jax.random.PRNGKey(123)):
    keys = jax.random.split(key, len(layers))
    params = []
    for i in range(len(layers) - 2):
        W = jax.random.normal(keys[i], shape=(layers[i], layers[i+1], degree + 1)) / jnp.sqrt(layers[i] * (degree + 1))
        g = jax.random.normal(keys[i], shape=(layers[i+1],))
        params.append({'W': W, 'g': g})
    W = jax.random.normal(keys[-1], shape=(layers[-2], layers[-1])) / jnp.sqrt(layers[-2])
    B = jax.random.normal(keys[-1], shape=(layers[-1],))
    params.append({'W': W, 'B': B})
    return params

# === 4. Forward pass ===
def fwd(params, t, activation=jax.nn.tanh):
    t = 0.01 * t
    X = t.reshape((-1, 1))
    *hidden, last = params
    for layer in hidden:
        W = layer['W']
        g = layer['g']
        degree = W.shape[-1] - 1
        X = activation(X)
        X_stack = chebyshev_recursive(X, degree)
        X = jnp.einsum("bid,iod->bo", X_stack, W)
    if X.shape[1] > 1:
        X = X[:, 0:1]
        # X = g * X
    return X# @ last['W'] #+ last['B']

# === 5. Loss ===
def mse_loss(params, t, y_true):
    y_pred = fwd(params, t)
    return jnp.mean((y_pred[:, 0] - y_true) ** 2)

# === 6. FFT Spectrum ===
def compute_fourier_spectrum(signal, dt):
    n = len(signal)
    freqs = np.fft.fftfreq(n, d=dt)
    fft_vals = np.fft.fft(signal)
    magnitude = np.abs(fft_vals)
    return freqs[:n // 2], magnitude[:n // 2]

# === 7. Setup ===
layers = [1, 64, 64, 1]
degree = 5
lr = 1e-4
epochs = 40000
log_epochs = list(range(0, epochs + 1, 1000))

t = jnp.linspace(0, 300, 301)
y_true = target_function(t)

params = init_params_kan2(layers, degree)
optimizer = optax.adam(lr)
opt_state = optimizer.init(params)

# === 8. Training step ===
@jax.jit
def train_step(params, opt_state, t, y_true):
    loss, grads = jax.value_and_grad(mse_loss)(params, t, y_true)
    updates, opt_state = optimizer.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss

# === 9. Training loop ===
predictions = {}
loss_history = []

for epoch in range(epochs + 1):
    params, opt_state, loss = train_step(params, opt_state, t, y_true)
    loss_history.append(loss)
    if epoch in log_epochs:
        predictions[epoch] = np.array(fwd(params, t)[:, 0])

# === 10. Animation: Signal vs. Prediction + Spectrum ===
plt.switch_backend("Agg")
dt = float(t[1] - t[0])
log_epochs = sorted(predictions.keys())

freqs = np.fft.fftfreq(len(t), d=dt)
mask = freqs >= 0
freqs_pos = freqs[mask]
fft_true = np.fft.fft(np.array(y_true))
amp_true = np.abs(fft_true)[mask]

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
plt.tight_layout()

def animate(i):
    epoch = log_epochs[i]
    y_pred = predictions[epoch]
    amp_pred = np.abs(np.fft.fft(y_pred))[mask]

    axs[0].clear()
    axs[0].plot(t, y_true, label='True', color='black')
    axs[0].plot(t, y_pred, '--', label=f'Pred', color='red')
    axs[0].set_title(f"Signal Prediction at Epoch {epoch}")
    axs[0].set_xlabel("Time")
    axs[0].set_ylabel("Signal")
    axs[0].legend()
    axs[0].grid(True)

    axs[1].clear()
    axs[1].plot(freqs_pos, amp_true, label='True Spectrum', color='black')
    axs[1].plot(freqs_pos, amp_pred, '--', label='Predicted Spectrum', color='red')
    axs[1].set_title(f"Frequency Spectrum at Epoch {epoch}")
    axs[1].set_xlabel("Frequency (Hz)")
    axs[1].set_ylabel("Amplitude")
    axs[1].set_xlim(0, 0.2)
    axs[1].set_yscale("log")
    axs[1].grid(True, which="both", ls="--", alpha=0.5)
    axs[1].legend()

anim = FuncAnimation(fig, animate, frames=len(log_epochs), interval=500)

# Save animation
anim.save("training_evolution_cKAN.gif", writer=PillowWriter(fps=2))
anim.save("training_evolution_cKAN.mp4", fps=2, extra_args=['-vcodec', 'libx264'])

print("Saved animation: training_evolution.gif, training_evolution.mp4")


Saved animation: training_evolution.gif, training_evolution.mp4


In [ ]:
import jax
import jax.numpy as jnp
import optax
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

# === 1. Piecewise target function ===
def target_function(x):
    left = 5.0 + jnp.sum(jnp.stack([jnp.sin(k * x) for k in range(1, 5)]), axis=0)
    right = jnp.cos(10 * x)
    return jnp.where(x < 0, left, right)

# === 2. Chebyshev recursive basis ===
def chebyshev_recursive(x, degree):
    T_n_minus_2 = x * 0 + 1
    T_n_minus_1 = x
    result = [T_n_minus_2, T_n_minus_1]
    for n in range(2, degree + 1):
        T_n = 2 * x * T_n_minus_1 - T_n_minus_2
        result.append(T_n)
        T_n_minus_2, T_n_minus_1 = T_n_minus_1, T_n
    return jnp.stack(result, axis=-1)

# === 3. KAN parameter initialization ===
def init_params_kan2(layers, degree, key=jax.random.PRNGKey(123)):
    keys = jax.random.split(key, len(layers))
    params = []
    for i in range(len(layers) - 2):
        W = jax.random.normal(keys[i], shape=(layers[i], layers[i+1], degree + 1)) / jnp.sqrt(layers[i] * (degree + 1))
        g = jax.random.normal(keys[i], shape=(layers[i+1],))
        params.append({'W': W, 'g': g})
    W = jax.random.normal(keys[-1], shape=(layers[-2], layers[-1])) / jnp.sqrt(layers[-2])
    B = jax.random.normal(keys[-1], shape=(layers[-1],))
    params.append({'W': W, 'B': B})
    return params

# === 4. Forward pass ===
def fwd(params, x, activation=jax.nn.tanh):
    X = x.reshape((-1, 1))
    *hidden, last = params
    for layer in hidden:
        W = layer['W']
        g = layer['g']
        degree = W.shape[-1] - 1
        X_stack = chebyshev_recursive(X, degree)
        X = jnp.einsum("bid,iod->bo", X_stack, W)
        X = g * X
        X = activation(X)
    return X @ last['W'] + last['B']

# === 5. MSE loss ===
def mse_loss(params, x, y_true):
    y_pred = fwd(params, x)
    return jnp.mean((y_pred[:, 0] - y_true) ** 2)

# === 6. FFT Spectrum ===
def compute_fourier_spectrum(signal, dx):
    n = len(signal)
    freqs = np.fft.fftfreq(n, d=dx)
    fft_vals = np.fft.fft(signal)
    magnitude = np.abs(fft_vals)
    return freqs[:n // 2], magnitude[:n // 2]

# === 7. Setup ===
layers = [1, 64, 64, 1]
degree = 5
lr = 1e-4
epochs = 40000
log_epochs = list(range(0, epochs + 1, 1000))

x = jnp.linspace(-jnp.pi, jnp.pi, 80)  # smaller dataset, centered domain
y_true = target_function(x)

params = init_params_kan2(layers, degree)
optimizer = optax.adam(lr)
opt_state = optimizer.init(params)

# === 8. Training step ===
@jax.jit
def train_step(params, opt_state, x, y_true):
    loss, grads = jax.value_and_grad(mse_loss)(params, x, y_true)
    updates, opt_state = optimizer.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss

# === 9. Training loop ===
predictions = {}
loss_history = []

for epoch in range(epochs + 1):
    params, opt_state, loss = train_step(params, opt_state, x, y_true)
    loss_history.append(loss)
    if epoch in log_epochs:
        predictions[epoch] = np.array(fwd(params, x)[:, 0])

# === 10. Animation: Signal vs. Prediction + Spectrum ===
plt.switch_backend("Agg")
dx = float(x[1] - x[0])
log_epochs = sorted(predictions.keys())

freqs = np.fft.fftfreq(len(x), d=dx)
mask = freqs >= 0
freqs_pos = freqs[mask]
fft_true = np.fft.fft(np.array(y_true))
amp_true = np.abs(fft_true)[mask]

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
plt.tight_layout()

def animate(i):
    epoch = log_epochs[i]
    y_pred = predictions[epoch]
    amp_pred = np.abs(np.fft.fft(y_pred))[mask]

    axs[0].clear()
    axs[0].plot(x, y_true, label='True', color='black')
    axs[0].plot(x, y_pred, '--', label=f'Pred', color='red')
    axs[0].set_title(f"Signal Prediction at Epoch {epoch}")
    axs[0].set_xlabel("x")
    axs[0].set_ylabel("y")
    axs[0].legend()
    axs[0].grid(True)

    axs[1].clear()
    axs[1].plot(freqs_pos, amp_true, label='True Spectrum', color='black')
    axs[1].plot(freqs_pos, amp_pred, '--', label='Predicted Spectrum', color='red')
    axs[1].set_title(f"Frequency Spectrum at Epoch {epoch}")
    axs[1].set_xlabel("f")
    axs[1].set_ylabel("Amplitude")
    axs[1].set_xlim(0, 6)
    axs[1].set_yscale("log")
    axs[1].grid(True, which="both", ls="--", alpha=0.5)
    axs[1].legend()

anim = FuncAnimation(fig, animate, frames=len(log_epochs), interval=500)

anim.save("gkan_disc.gif", writer=PillowWriter(fps=2))
print("Saved animation: gkan_disc.gif")


Saved animation: training_evolution_piecewise.gif


In [ ]:
import jax
import jax.numpy as jnp
import optax
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

# === 1. Piecewise target function ===
def target_function(x):
    left = 5.0 + jnp.sum(jnp.stack([jnp.sin(k * x) for k in range(1, 5)]), axis=0)
    right = jnp.cos(10 * x)
    return jnp.where(x < 0, left, right)

# === 2. Chebyshev recursive basis ===
def chebyshev_recursive(x, degree):
    T_n_minus_2 = x * 0 + 1
    T_n_minus_1 = x
    result = [T_n_minus_2, T_n_minus_1]
    for n in range(2, degree + 1):
        T_n = 2 * x * T_n_minus_1 - T_n_minus_2
        result.append(T_n)
        T_n_minus_2, T_n_minus_1 = T_n_minus_1, T_n
    return jnp.stack(result, axis=-1)

# === 3. KAN parameter initialization ===
def init_params_kan2(layers, degree, key=jax.random.PRNGKey(123)):
    keys = jax.random.split(key, len(layers))
    params = []
    for i in range(len(layers) - 2):
        W = jax.random.normal(keys[i], shape=(layers[i], layers[i+1], degree + 1)) / jnp.sqrt(layers[i] * (degree + 1))
        g = jax.random.normal(keys[i], shape=(layers[i+1],))
        params.append({'W': W, 'g': g})
    W = jax.random.normal(keys[-1], shape=(layers[-2], layers[-1])) / jnp.sqrt(layers[-2])
    B = jax.random.normal(keys[-1], shape=(layers[-1],))
    params.append({'W': W, 'B': B})
    return params

# === 4. Forward pass ===
def fwd(params, x, activation=jax.nn.tanh):
    X = x.reshape((-1, 1))
    *hidden, last = params
    for layer in hidden:
        W = layer['W']
        g = layer['g']
        degree = W.shape[-1] - 1
        X = activation(X)
        X_stack = chebyshev_recursive(X, degree)
        X = jnp.einsum("bid,iod->bo", X_stack, W)
        # X = g * X
        X = activation(X)
    return X @ last['W'] + last['B']

# === 5. MSE loss ===
def mse_loss(params, x, y_true):
    y_pred = fwd(params, x)
    return jnp.mean((y_pred[:, 0] - y_true) ** 2)

# === 6. FFT Spectrum ===
def compute_fourier_spectrum(signal, dx):
    n = len(signal)
    freqs = np.fft.fftfreq(n, d=dx)
    fft_vals = np.fft.fft(signal)
    magnitude = np.abs(fft_vals)
    return freqs[:n // 2], magnitude[:n // 2]

# === 7. Setup ===
layers = [1, 64, 64, 1]
degree = 5
lr = 1e-4
epochs = 40000
log_epochs = list(range(0, epochs + 1, 1000))

x = jnp.linspace(-jnp.pi, jnp.pi, 80)  # smaller dataset, centered domain
y_true = target_function(x)

params = init_params_kan2(layers, degree)
optimizer = optax.adam(lr)
opt_state = optimizer.init(params)

# === 8. Training step ===
@jax.jit
def train_step(params, opt_state, x, y_true):
    loss, grads = jax.value_and_grad(mse_loss)(params, x, y_true)
    updates, opt_state = optimizer.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss

# === 9. Training loop ===
predictions = {}
loss_history = []

for epoch in range(epochs + 1):
    params, opt_state, loss = train_step(params, opt_state, x, y_true)
    loss_history.append(loss)
    if epoch in log_epochs:
        predictions[epoch] = np.array(fwd(params, x)[:, 0])

# === 10. Animation: Signal vs. Prediction + Spectrum ===
plt.switch_backend("Agg")
dx = float(x[1] - x[0])
log_epochs = sorted(predictions.keys())

freqs = np.fft.fftfreq(len(x), d=dx)
mask = freqs >= 0
freqs_pos = freqs[mask]
fft_true = np.fft.fft(np.array(y_true))
amp_true = np.abs(fft_true)[mask]

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
plt.tight_layout()

def animate(i):
    epoch = log_epochs[i]
    y_pred = predictions[epoch]
    amp_pred = np.abs(np.fft.fft(y_pred))[mask]

    axs[0].clear()
    axs[0].plot(x, y_true, label='True', color='black')
    axs[0].plot(x, y_pred, '--', label=f'Pred', color='red')
    axs[0].set_title(f"Signal Prediction at Epoch {epoch}")
    axs[0].set_xlabel("x")
    axs[0].set_ylabel("y")
    axs[0].legend()
    axs[0].grid(True)

    axs[1].clear()
    axs[1].plot(freqs_pos, amp_true, label='True Spectrum', color='black')
    axs[1].plot(freqs_pos, amp_pred, '--', label='Predicted Spectrum', color='red')
    axs[1].set_title(f"Frequency Spectrum at Epoch {epoch}")
    axs[1].set_xlabel("f")
    axs[1].set_ylabel("Amplitude")
    axs[1].set_xlim(0, 6)
    axs[1].set_yscale("log")
    axs[1].grid(True, which="both", ls="--", alpha=0.5)
    axs[1].legend()

anim = FuncAnimation(fig, animate, frames=len(log_epochs), interval=500)

anim.save("tanh-ckan_disc.gif", writer=PillowWriter(fps=2))
print("Saved animation: tanh-kan_disc.gif")


Saved animation: tanh-kan_disc.gif


In [ ]:
import jax
import jax.numpy as jnp
import optax
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

# === 1. Piecewise target function ===
def target_function(x):
    left = 5.0 + jnp.sum(jnp.stack([jnp.sin(k * x) for k in range(1, 5)]), axis=0)
    right = jnp.cos(10 * x)
    return jnp.where(x < 0, left, right)

# === 2. Chebyshev recursive basis ===
def chebyshev_recursive(x, degree):
    T_n_minus_2 = x * 0 + 1
    T_n_minus_1 = x
    result = [T_n_minus_2, T_n_minus_1]
    for n in range(2, degree + 1):
        T_n = 2 * x * T_n_minus_1 - T_n_minus_2
        result.append(T_n)
        T_n_minus_2, T_n_minus_1 = T_n_minus_1, T_n
    return jnp.stack(result, axis=-1)

# === 3. KAN parameter initialization ===
def init_params_kan2(layers, degree, key=jax.random.PRNGKey(123)):
    keys = jax.random.split(key, len(layers))
    params = []
    for i in range(len(layers) - 2):
        W = jax.random.normal(keys[i], shape=(layers[i], layers[i+1], degree + 1)) / jnp.sqrt(layers[i] * (degree + 1))
        g = jax.random.normal(keys[i], shape=(layers[i+1],))
        params.append({'W': W, 'g': g})
    W = jax.random.normal(keys[-1], shape=(layers[-2], layers[-1])) / jnp.sqrt(layers[-2])
    B = jax.random.normal(keys[-1], shape=(layers[-1],))
    params.append({'W': W, 'B': B})
    return params

# === 4. Forward pass ===
def fwd(params, t, activation=jax.nn.tanh):
    # t = 0.01 * t
    X = t.reshape((-1, 1))
    *hidden, last = params
    for layer in hidden:
        W = layer['W']
        g = layer['g']
        degree = W.shape[-1] - 1
        X = activation(X)
        X_stack = chebyshev_recursive(X, degree)
        X = jnp.einsum("bid,iod->bo", X_stack, W)
    if X.shape[1] > 1:
        X = X[:, 0:1]
        # X = g * X
    return X# @ last['W'] #+ last['B']

# === 5. MSE loss ===
def mse_loss(params, x, y_true):
    y_pred = fwd(params, x)
    return jnp.mean((y_pred[:, 0] - y_true) ** 2)

# === 6. FFT Spectrum ===
def compute_fourier_spectrum(signal, dx):
    n = len(signal)
    freqs = np.fft.fftfreq(n, d=dx)
    fft_vals = np.fft.fft(signal)
    magnitude = np.abs(fft_vals)
    return freqs[:n // 2], magnitude[:n // 2]

# === 7. Setup ===
layers = [1, 64, 64, 1]
degree = 5
lr = 1e-4
epochs = 40000
log_epochs = list(range(0, epochs + 1, 1000))

x = jnp.linspace(-jnp.pi, jnp.pi, 80)  # smaller dataset, centered domain
y_true = target_function(x)

params = init_params_kan2(layers, degree)
optimizer = optax.adam(lr)
opt_state = optimizer.init(params)

# === 8. Training step ===
@jax.jit
def train_step(params, opt_state, x, y_true):
    loss, grads = jax.value_and_grad(mse_loss)(params, x, y_true)
    updates, opt_state = optimizer.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss

# === 9. Training loop ===
predictions = {}
loss_history = []

for epoch in range(epochs + 1):
    params, opt_state, loss = train_step(params, opt_state, x, y_true)
    loss_history.append(loss)
    if epoch in log_epochs:
        predictions[epoch] = np.array(fwd(params, x)[:, 0])

# === 10. Animation: Signal vs. Prediction + Spectrum ===
plt.switch_backend("Agg")
dx = float(x[1] - x[0])
log_epochs = sorted(predictions.keys())

freqs = np.fft.fftfreq(len(x), d=dx)
mask = freqs >= 0
freqs_pos = freqs[mask]
fft_true = np.fft.fft(np.array(y_true))
amp_true = np.abs(fft_true)[mask]

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
plt.tight_layout()

def animate(i):
    epoch = log_epochs[i]
    y_pred = predictions[epoch]
    amp_pred = np.abs(np.fft.fft(y_pred))[mask]

    axs[0].clear()
    axs[0].plot(x, y_true, label='True', color='black')
    axs[0].plot(x, y_pred, '--', label=f'Pred', color='red')
    axs[0].set_title(f"Signal Prediction at Epoch {epoch}")
    axs[0].set_xlabel("x")
    axs[0].set_ylabel("y")
    axs[0].legend()
    axs[0].grid(True)

    axs[1].clear()
    axs[1].plot(freqs_pos, amp_true, label='True Spectrum', color='black')
    axs[1].plot(freqs_pos, amp_pred, '--', label='Predicted Spectrum', color='red')
    axs[1].set_title(f"Frequency Spectrum at Epoch {epoch}")
    axs[1].set_xlabel("f")
    axs[1].set_ylabel("Amplitude")
    axs[1].set_xlim(0, 6)
    axs[1].set_yscale("log")
    axs[1].grid(True, which="both", ls="--", alpha=0.5)
    axs[1].legend()

anim = FuncAnimation(fig, animate, frames=len(log_epochs), interval=500)

anim.save("ckan_disc.gif", writer=PillowWriter(fps=2))
print("Saved animation: ckan_disc.gif")


Saved animation: ckan_disc.gif


In [ ]:
import jax
import jax.numpy as jnp
import optax
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

# === 1. Target function with multiple frequencies ===
def target_function(t):
    return (
        jnp.sin(2 * jnp.pi * 0.01 * t)
        + 0.5 * jnp.sin(2 * jnp.pi * 0.05 * t)
        + 0.2 * jnp.sin(2 * jnp.pi * 0.1 * t)
    )

# ==========================================================
#                 SIMPLE MLP (REPLACES KAN)
# ==========================================================
def init_params_mlp(layers, key=jax.random.PRNGKey(0)):
    params = []
    keys = jax.random.split(key, len(layers))

    for i in range(len(layers) - 1):
        W = jax.random.normal(keys[i], (layers[i], layers[i+1])) / jnp.sqrt(layers[i])
        B = jnp.zeros((layers[i+1],))
        params.append({"W": W, "B": B})

    return params


def fwd_mlp(params, t, activation=jax.nn.tanh):
    x = 0.01 * t.reshape((-1, 1))

    for layer in params[:-1]:
        x = activation(x @ layer["W"] + layer["B"])

    last = params[-1]
    out = x @ last["W"] + last["B"]  # (N,1)
    return out


# === 3. Loss ===
def mse_loss(params, t, y_true):
    y_pred = fwd_mlp(params, t)
    return jnp.mean((y_pred[:, 0] - y_true) ** 2)


# === 4. FFT Spectrum ===
def compute_fourier_spectrum(signal, dt):
    n = len(signal)
    freqs = np.fft.fftfreq(n, d=dt)
    fft_vals = np.fft.fft(signal)
    magnitude = np.abs(fft_vals)
    return freqs[: n // 2], magnitude[: n // 2]


# === 5. Setup ===
layers = [1, 64, 64, 1]   # Standard MLP architecture
lr = 1e-4
epochs = 40000
log_epochs = list(range(0, epochs + 1, 1000))

t = jnp.linspace(0, 300, 301)
y_true = target_function(t)

params = init_params_mlp(layers)
optimizer = optax.adam(lr)
opt_state = optimizer.init(params)


# === 6. Training step ===
@jax.jit
def train_step(params, opt_state, t, y_true):
    loss, grads = jax.value_and_grad(mse_loss)(params, t, y_true)
    updates, opt_state = optimizer.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss


# === 7. Training loop ===
predictions = {}
loss_history = []

for epoch in range(epochs + 1):
    params, opt_state, loss = train_step(params, opt_state, t, y_true)
    loss_history.append(loss)
    if epoch in log_epochs:
        predictions[epoch] = np.array(fwd_mlp(params, t)[:, 0])


# === 8. Animation ===
plt.switch_backend("Agg")
dt = float(t[1] - t[0])
log_epochs = sorted(predictions.keys())

freqs = np.fft.fftfreq(len(t), d=dt)
mask = freqs >= 0
freqs_pos = freqs[mask]
amp_true = np.abs(np.fft.fft(np.array(y_true)))[mask]

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
plt.tight_layout()

def animate(i):
    epoch = log_epochs[i]
    y_pred = predictions[epoch]
    amp_pred = np.abs(np.fft.fft(y_pred))[mask]

    axs[0].clear()
    axs[0].plot(t, y_true, label='True', color='black')
    axs[0].plot(t, y_pred, '--', label='Pred', color='red')
    axs[0].set_title(f"Signal Prediction at Epoch {epoch}")
    axs[0].set_xlabel("Time")
    axs[0].set_ylabel("Signal")
    axs[0].legend()
    axs[0].grid(True)

    axs[1].clear()
    axs[1].plot(freqs_pos, amp_true, label='True Spectrum', color='black')
    axs[1].plot(freqs_pos, amp_pred, '--', label='Pred Spectrum', color='red')
    axs[1].set_title(f"Frequency Spectrum at Epoch {epoch}")
    axs[1].set_xlabel("Frequency")
    axs[1].set_ylabel("Amplitude (log)")
    axs[1].set_xlim(0, 0.2)
    axs[1].set_yscale("log")
    axs[1].grid(True, which="both", ls="--", alpha=0.5)
    axs[1].legend()

anim = FuncAnimation(fig, animate, frames=len(log_epochs), interval=500)

anim.save("training_evolution_mlp.gif", writer=PillowWriter(fps=2))
anim.save("training_evolution_mlp.mp4", fps=2, extra_args=['-vcodec', 'libx264'])

print("Saved animation: training_evolution_mlp.gif, training_evolution_mlp.mp4")


Saved animation: training_evolution_mlp.gif, training_evolution_mlp.mp4


In [9]:
import jax
import jax.numpy as jnp
import optax
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# 1. Target function (multi-frequency signal)
# ============================================================
def target_function(t):
    return (
        jnp.sin(2 * jnp.pi * 0.01 * t)
        + 0.5 * jnp.sin(2 * jnp.pi * 0.05 * t)
        + 0.2 * jnp.sin(2 * jnp.pi * 0.1 * t)
    )

# ============================================================
# 2. SIREN definition
# ============================================================
def siren_layer_init(key, in_dim, out_dim, w0, is_first):
    if is_first:
        bound = 1.0 / in_dim
    else:
        bound = jnp.sqrt(6.0 / in_dim) / w0
    W = jax.random.uniform(key, (in_dim, out_dim), minval=-bound, maxval=bound)
    b = jnp.zeros((out_dim,))
    return {"W": W, "b": b}


def init_siren(layers, w0=10.0, key=jax.random.PRNGKey(0)):
    params = []
    keys = jax.random.split(key, len(layers))
    for i in range(len(layers) - 1):
        params.append(
            siren_layer_init(
                keys[i],
                layers[i],
                layers[i + 1],
                w0=w0,
                is_first=(i == 0),
            )
        )
    return params


def fwd_siren(params, t, w0=10.0):
    x = t.reshape((-1, 1))
    for layer in params[:-1]:
        x = jnp.sin(w0 * (x @ layer["W"] + layer["b"]))
    last = params[-1]
    return x @ last["W"] + last["b"]

# ============================================================
# 3. Loss
# ============================================================
def mse_loss(params, t, y_true):
    y_pred = fwd_siren(params, t)
    return jnp.mean((y_pred[:, 0] - y_true) ** 2)

# ============================================================
# 4. Setup
# ============================================================
layers = [1, 64, 64, 1]
w0 = 10.0
epochs = 40000
lr = 1e-4

t = jnp.linspace(0, 300, 301)
y_true = target_function(t)

params = init_siren(layers, w0=w0)

optimizer = optax.adam(lr)
opt_state = optimizer.init(params)

@jax.jit
def train_step(params, opt_state):
    loss, grads = jax.value_and_grad(mse_loss)(params, t, y_true)
    updates, opt_state = optimizer.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss

# ============================================================
# 5. Training loop (store snapshots)
# ============================================================
epochs_to_plot = [0, 5000, 15000, 25000, 35000, 40000]
predictions = {}

for epoch in range(epochs + 1):
    params, opt_state, loss = train_step(params, opt_state)

    if epoch in epochs_to_plot:
        predictions[epoch] = np.array(fwd_siren(params, t)[:, 0])

    if epoch % 5000 == 0:
        print(f"[Adam {epoch:6d}] loss = {loss:.3e}")

# ============================================================
# 6. FFT helper
# ============================================================
def fft_mag(signal, dt):
    fft_vals = np.fft.fft(signal)
    freqs = np.fft.fftfreq(len(signal), d=dt)
    mask = freqs >= 0
    return freqs[mask], np.abs(fft_vals[mask])

# ============================================================
# 7. Spectral Learning Evolution Figure (2 × N)
# ============================================================
dt = float(t[1] - t[0])
freqs_true, amp_true = fft_mag(np.array(y_true), dt)

ncols = len(epochs_to_plot)
fig, axes = plt.subplots(
    2, ncols,
    figsize=(3.2 * ncols, 6),
    constrained_layout=True
)

for j, ep in enumerate(epochs_to_plot):
    y_pred = predictions[ep]

    # --- Top row: physical space ---
    ax = axes[0, j]
    ax.plot(t, y_true, color="black", lw=1.8)
    ax.plot(t, y_pred, "r--", lw=1.8)
    ax.set_title(f"Epoch {ep}", fontsize=12)
    ax.grid(True, alpha=0.3)

    if j == 0:
        ax.set_ylabel(r"$u(x)$", fontsize=12)
    else:
        ax.set_yticklabels([])

    # --- Bottom row: Fourier spectrum ---
    ax = axes[1, j]
    freqs_pred, amp_pred = fft_mag(y_pred, dt)

    ax.semilogy(freqs_true, amp_true, color="black", lw=1.8)
    ax.semilogy(freqs_pred, amp_pred, "r--", lw=1.8)
    ax.set_xlim(0, 0.5)
    ax.grid(True, which="both", ls="--", alpha=0.4)

    if j == 0:
        ax.set_ylabel(r"$|\hat{u}(k)|$", fontsize=12)
    else:
        ax.set_yticklabels([])

    ax.set_xlabel(r"$k$", fontsize=11)

# Global title
fig.suptitle(
    "Spectral Learning Pattern Evolution (SIREN)",
    fontsize=16,
    y=1.03
)

plt.savefig(
    "spectral_learning_evolution_siren.pdf",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

print("✓ Saved figure: spectral_learning_evolution_siren.png")


[Adam      0] loss = 6.554e-01
[Adam   5000] loss = 1.071e-05
[Adam  10000] loss = 2.722e-06
[Adam  15000] loss = 1.204e-05
[Adam  20000] loss = 4.595e-06
[Adam  25000] loss = 6.097e-06
[Adam  30000] loss = 3.363e-06
[Adam  35000] loss = 9.194e-07
[Adam  40000] loss = 3.339e-06
✓ Saved figure: spectral_learning_evolution_siren.png


In [2]:
import jax
import jax.numpy as jnp
import optax
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

# ============================================
# 1. Target function (unchanged)
# ============================================
def target_function(t):
    return (
        jnp.sin(2 * jnp.pi * 0.01 * t)
        + 0.5 * jnp.sin(2 * jnp.pi * 0.05 * t)
        + 0.2 * jnp.sin(2 * jnp.pi * 0.1 * t)
    )


# ============================================
# 2. SIREN network
# ============================================
def siren_layer_init(key, in_dim, out_dim, w0, is_first):
    if is_first:
        bound = 1.0 / in_dim
    else:
        bound = jnp.sqrt(6.0 / in_dim) / w0
    W = jax.random.uniform(key, (in_dim, out_dim), minval=-bound, maxval=bound)
    b = jnp.zeros((out_dim,))
    return {"W": W, "b": b}


def init_siren(layers, w0=30.0, key=jax.random.PRNGKey(0)):
    params = []
    keys = jax.random.split(key, len(layers))
    for i in range(len(layers) - 1):
        params.append(
            siren_layer_init(
                keys[i],
                layers[i],
                layers[i + 1],
                w0=w0,
                is_first=(i == 0),
            )
        )
    return params


def fwd_siren(params, t, w0=30.0):
    x = t.reshape((-1, 1))
    for layer in params[:-1]:
        x = jnp.sin(w0 * (x @ layer["W"] + layer["b"]))
    last = params[-1]
    return x @ last["W"] + last["b"]


# ============================================
# 3. Loss
# ============================================
def mse_loss(params, t, y_true):
    y_pred = fwd_siren(params, t)
    return jnp.mean((y_pred[:, 0] - y_true) ** 2)


# ============================================
# 4. Setup
# ============================================
layers = [1, 64, 64, 1]
epochs = 40000
lr = 1e-4

t = jnp.linspace(0, 300, 301)
y_true = target_function(t)

params = init_siren(layers, w0=10.0)

optimizer = optax.adam(lr)
opt_state = optimizer.init(params)


# ============================================
# 5. Training step
# ============================================
@jax.jit
def train_step(params, opt_state):
    loss, grads = jax.value_and_grad(mse_loss)(params, t, y_true)
    updates, opt_state = optimizer.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss


# ============================================
# 6. Training loop
# ============================================
log_epochs = list(range(0, epochs + 1, 1000))
predictions = {}
loss_history = []

for epoch in range(epochs + 1):
    params, opt_state, loss = train_step(params, opt_state)
    loss_history.append(loss)

    if epoch in log_epochs:
        predictions[epoch] = np.array(fwd_siren(params, t)[:, 0])

    if epoch % 1000 == 0:
        print(f"[Adam {epoch:6d}] loss = {loss:.3e}")


# ============================================
# 7. Animation (unchanged)
# ============================================
plt.switch_backend("Agg")

dt = float(t[1] - t[0])
freqs = np.fft.fftfreq(len(t), d=dt)
mask = freqs >= 0
freqs_pos = freqs[mask]
amp_true = np.abs(np.fft.fft(np.array(y_true)))[mask]

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
plt.tight_layout()

def animate(i):
    epoch = log_epochs[i]
    y_pred = predictions[epoch]
    amp_pred = np.abs(np.fft.fft(y_pred))[mask]

    axs[0].clear()
    axs[0].plot(t, y_true, label="True", color="black")
    axs[0].plot(t, y_pred, "--", label="Pred", color="red")
    axs[0].set_title(f"Prediction at Epoch {epoch}")
    axs[0].legend()
    axs[0].grid(True)

    axs[1].clear()
    axs[1].plot(freqs_pos, amp_true, label="True", color="black")
    axs[1].plot(freqs_pos, amp_pred, "--", label="Pred", color="red")
    axs[1].set_xlim(0, 0.2)
    axs[1].set_yscale("log")
    axs[1].grid(True, which="both", ls="--")
    axs[1].legend()

anim = FuncAnimation(fig, animate, frames=len(log_epochs), interval=500)
anim.save("training_evolution_siren_adam.gif", writer=PillowWriter(fps=2))
anim.save("training_evolution_siren_adam.mp4", fps=2)

print("Saved animation: training_evolution_siren_adam.gif / .mp4")


[Adam      0] loss = 6.633e-01
[Adam   1000] loss = 1.864e-07
[Adam   2000] loss = 2.953e-08
[Adam   3000] loss = 8.385e-08
[Adam   4000] loss = 1.842e-05
[Adam   5000] loss = 4.098e-05
[Adam   6000] loss = 5.999e-08
[Adam   7000] loss = 1.708e-07
[Adam   8000] loss = 2.050e-04
[Adam   9000] loss = 2.375e-05
[Adam  10000] loss = 7.903e-08
[Adam  11000] loss = 1.162e-07
[Adam  12000] loss = 8.706e-05
[Adam  13000] loss = 6.718e-05
[Adam  14000] loss = 9.251e-08
[Adam  15000] loss = 3.720e-07
[Adam  16000] loss = 5.317e-06
[Adam  17000] loss = 3.678e-04
[Adam  18000] loss = 7.692e-08
[Adam  19000] loss = 3.252e-07
[Adam  20000] loss = 1.114e-04
[Adam  21000] loss = 1.905e-04
[Adam  22000] loss = 1.023e-07
[Adam  23000] loss = 1.691e-04
[Adam  24000] loss = 2.174e-05
[Adam  25000] loss = 1.660e-04
[Adam  26000] loss = 1.328e-07
[Adam  27000] loss = 4.823e-05
[Adam  28000] loss = 2.520e-04
[Adam  29000] loss = 1.568e-06
[Adam  30000] loss = 1.195e-07
[Adam  31000] loss = 7.674e-05
[Adam  3

In [ ]:
import jax
import jax.numpy as jnp
import optax
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

# === 1. Piecewise target function ===
def target_function(x):
    left = 5.0 + jnp.sum(jnp.stack([jnp.sin(k * x) for k in range(1, 5)]), axis=0)
    right = jnp.cos(10 * x)
    return jnp.where(x < 0, left, right)


# ==========================================================
#               SIMPLE MLP (REPLACES KAN)
# ==========================================================
def init_params_mlp(layers, key=jax.random.PRNGKey(0)):
    params = []
    keys = jax.random.split(key, len(layers))

    for i in range(len(layers) - 1):
        W = jax.random.normal(keys[i], (layers[i], layers[i+1])) / jnp.sqrt(layers[i])
        B = jnp.zeros((layers[i+1],))
        params.append({"W": W, "B": B})

    return params


def fwd(params, x, activation=jax.nn.tanh):
    X = x.reshape((-1, 1))

    for layer in params[:-1]:
        X = activation(X @ layer["W"] + layer["B"])

    last = params[-1]
    X = X @ last["W"] + last["B"]
    return X


# === 5. MSE loss ===
def mse_loss(params, x, y_true):
    y_pred = fwd(params, x)
    return jnp.mean((y_pred[:, 0] - y_true) ** 2)


# === 6. FFT Spectrum ===
def compute_fourier_spectrum(signal, dx):
    n = len(signal)
    freqs = np.fft.fftfreq(n, d=dx)
    fft_vals = np.fft.fft(signal)
    magnitude = np.abs(fft_vals)
    return freqs[:n // 2], magnitude[:n // 2]


# === 7. Setup ===
layers = [1, 32, 32, 1]
lr = 1e-4
epochs = 40000
log_epochs = list(range(0, epochs + 1, 1000))

x = jnp.linspace(-jnp.pi, jnp.pi, 80)
y_true = target_function(x)

params = init_params_mlp(layers)
optimizer = optax.adam(lr)
opt_state = optimizer.init(params)


# === 8. Training step ===
@jax.jit
def train_step(params, opt_state, x, y_true):
    loss, grads = jax.value_and_grad(mse_loss)(params, x, y_true)
    updates, opt_state = optimizer.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss


# === 9. Training loop ===
predictions = {}
loss_history = []

for epoch in range(epochs + 1):
    params, opt_state, loss = train_step(params, opt_state, x, y_true)
    loss_history.append(loss)
    if epoch in log_epochs:
        predictions[epoch] = np.array(fwd(params, x)[:, 0])


# === 10. Animation: Signal vs. Prediction + Spectrum ===
plt.switch_backend("Agg")
dx = float(x[1] - x[0])
log_epochs = sorted(predictions.keys())

freqs = np.fft.fftfreq(len(x), d=dx)
mask = freqs >= 0
freqs_pos = freqs[mask]

fft_true = np.fft.fft(np.array(y_true))
amp_true = np.abs(fft_true)[mask]

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
plt.tight_layout()

def animate(i):
    epoch = log_epochs[i]
    y_pred = predictions[epoch]
    amp_pred = np.abs(np.fft.fft(y_pred))[mask]

    axs[0].clear()
    axs[0].plot(x, y_true, label='True', color='black')
    axs[0].plot(x, y_pred, '--', label=f'Pred', color='red')
    axs[0].set_title(f"Signal Prediction at Epoch {epoch}")
    axs[0].set_xlabel("x")
    axs[0].set_ylabel("y")
    axs[0].legend()
    axs[0].grid(True)

    axs[1].clear()
    axs[1].plot(freqs_pos, amp_true, label='True Spectrum', color='black')
    axs[1].plot(freqs_pos, amp_pred, '--', label='Predicted Spectrum', color='red')
    axs[1].set_title(f"Frequency Spectrum at Epoch {epoch}")
    axs[1].set_xlabel("f")
    axs[1].set_ylabel("Amplitude")
    axs[1].set_xlim(0, 6)
    axs[1].set_yscale("log")
    axs[1].grid(True, which="both", ls="--", alpha=0.5)
    axs[1].legend()

anim = FuncAnimation(fig, animate, frames=len(log_epochs), interval=500)

anim.save("mlp_piecewise.gif", writer=PillowWriter(fps=2))
print("Saved animation: mlp_piecewise.gif")


In [ ]:
import jax
import jax.numpy as jnp
import optax
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

# === 1. Piecewise target function ===
def target_function(x):
    left = 5.0 + jnp.sum(jnp.stack([jnp.sin(k * x) for k in range(1, 5)]), axis=0)
    right = jnp.cos(10 * x)
    return jnp.where(x < 0, left, right)


# ==========================================================
#               SIMPLE MLP (REPLACES KAN)
# ==========================================================
def init_params_mlp(layers, key=jax.random.PRNGKey(0)):
    params = []
    keys = jax.random.split(key, len(layers))

    for i in range(len(layers) - 1):
        W = jax.random.normal(keys[i], (layers[i], layers[i+1])) / jnp.sqrt(layers[i])
        B = jnp.zeros((layers[i+1],))
        params.append({"W": W, "B": B})

    return params


def fwd(params, x, activation=jax.nn.tanh):
    X = x.reshape((-1, 1))

    for layer in params[:-1]:
        X = activation(X @ layer["W"] + layer["B"])

    last = params[-1]
    X = X @ last["W"] + last["B"]
    return X


# === 5. MSE loss ===
def mse_loss(params, x, y_true):
    y_pred = fwd(params, x)
    return jnp.mean((y_pred[:, 0] - y_true) ** 2)


# === 6. FFT Spectrum ===
def compute_fourier_spectrum(signal, dx):
    n = len(signal)
    freqs = np.fft.fftfreq(n, d=dx)
    fft_vals = np.fft.fft(signal)
    magnitude = np.abs(fft_vals)
    return freqs[:n // 2], magnitude[:n // 2]


# === 7. Setup ===
layers = [1, 32, 32, 1]
lr = 1e-4
epochs = 40000
log_epochs = list(range(0, epochs + 1, 1000))

x = jnp.linspace(-jnp.pi, jnp.pi, 80)
y_true = target_function(x)

params = init_params_mlp(layers)
optimizer = optax.adam(lr)
opt_state = optimizer.init(params)


# === 8. Training step ===
@jax.jit
def train_step(params, opt_state, x, y_true):
    loss, grads = jax.value_and_grad(mse_loss)(params, x, y_true)
    updates, opt_state = optimizer.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss


# === 9. Training loop ===
predictions = {}
loss_history = []

for epoch in range(epochs + 1):
    params, opt_state, loss = train_step(params, opt_state, x, y_true)
    loss_history.append(loss)
    if epoch in log_epochs:
        predictions[epoch] = np.array(fwd(params, x)[:, 0])


# === 10. Animation: Signal vs. Prediction + Spectrum ===
plt.switch_backend("Agg")
dx = float(x[1] - x[0])
log_epochs = sorted(predictions.keys())

freqs = np.fft.fftfreq(len(x), d=dx)
mask = freqs >= 0
freqs_pos = freqs[mask]

fft_true = np.fft.fft(np.array(y_true))
amp_true = np.abs(fft_true)[mask]

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
plt.tight_layout()

def animate(i):
    epoch = log_epochs[i]
    y_pred = predictions[epoch]
    amp_pred = np.abs(np.fft.fft(y_pred))[mask]

    axs[0].clear()
    axs[0].plot(x, y_true, label='True', color='black')
    axs[0].plot(x, y_pred, '--', label=f'Pred', color='red')
    axs[0].set_title(f"Signal Prediction at Epoch {epoch}")
    axs[0].set_xlabel("x")
    axs[0].set_ylabel("y")
    axs[0].legend()
    axs[0].grid(True)

    axs[1].clear()
    axs[1].plot(freqs_pos, amp_true, label='True Spectrum', color='black')
    axs[1].plot(freqs_pos, amp_pred, '--', label='Predicted Spectrum', color='red')
    axs[1].set_title(f"Frequency Spectrum at Epoch {epoch}")
    axs[1].set_xlabel("f")
    axs[1].set_ylabel("Amplitude")
    axs[1].set_xlim(0, 6)
    axs[1].set_yscale("log")
    axs[1].grid(True, which="both", ls="--", alpha=0.5)
    axs[1].legend()

anim = FuncAnimation(fig, animate, frames=len(log_epochs), interval=500)

anim.save("mlp_piecewise.gif", writer=PillowWriter(fps=2))
print("Saved animation: mlp_piecewise.gif")


Saved animation: mlp_piecewise.gif


In [ ]:
import jax
import jax.numpy as jnp
import optax
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

# === 1. Piecewise target function ===
def target_function(x):
    left = 5.0 + jnp.sum(jnp.stack([jnp.sin(k * x) for k in range(1, 5)]), axis=0)
    right = jnp.cos(10 * x)
    return jnp.where(x < 0, left, right)


# ==========================================================
#               SIMPLE MLP (REPLACES KAN)
# ==========================================================
def init_params_mlp(layers, key=jax.random.PRNGKey(0)):
    params = []
    keys = jax.random.split(key, len(layers))

    for i in range(len(layers) - 1):
        W = jax.random.normal(keys[i], (layers[i], layers[i+1])) / jnp.sqrt(layers[i])
        B = jnp.zeros((layers[i+1],))
        params.append({"W": W, "B": B})

    return params


def fwd(params, x, activation=jax.nn.tanh):
    X = x.reshape((-1, 1))

    for layer in params[:-1]:
        X = activation(X @ layer["W"] + layer["B"])

    last = params[-1]
    X = X @ last["W"] + last["B"]
    return X


# === 5. MSE loss ===
def mse_loss(params, x, y_true):
    y_pred = fwd(params, x)
    return jnp.mean((y_pred[:, 0] - y_true) ** 2)


# === 6. FFT Spectrum ===
def compute_fourier_spectrum(signal, dx):
    n = len(signal)
    freqs = np.fft.fftfreq(n, d=dx)
    fft_vals = np.fft.fft(signal)
    magnitude = np.abs(fft_vals)
    return freqs[:n // 2], magnitude[:n // 2]


# === 7. Setup ===
layers = [1, 32, 32, 1]
lr = 1e-4
epochs = 40000
log_epochs = list(range(0, epochs + 1, 1000))

x = jnp.linspace(-jnp.pi, jnp.pi, 80)
y_true = target_function(x)

params = init_params_mlp(layers)
optimizer = optax.adam(lr)
opt_state = optimizer.init(params)


# === 8. Training step ===
@jax.jit
def train_step(params, opt_state, x, y_true):
    loss, grads = jax.value_and_grad(mse_loss)(params, x, y_true)
    updates, opt_state = optimizer.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss


# === 9. Training loop ===
predictions = {}
loss_history = []

for epoch in range(epochs + 1):
    params, opt_state, loss = train_step(params, opt_state, x, y_true)
    loss_history.append(loss)
    if epoch in log_epochs:
        predictions[epoch] = np.array(fwd(params, x)[:, 0])


# === 10. Animation: Signal vs. Prediction + Spectrum ===
plt.switch_backend("Agg")
dx = float(x[1] - x[0])
log_epochs = sorted(predictions.keys())

freqs = np.fft.fftfreq(len(x), d=dx)
mask = freqs >= 0
freqs_pos = freqs[mask]

fft_true = np.fft.fft(np.array(y_true))
amp_true = np.abs(fft_true)[mask]

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
plt.tight_layout()

def animate(i):
    epoch = log_epochs[i]
    y_pred = predictions[epoch]
    amp_pred = np.abs(np.fft.fft(y_pred))[mask]

    axs[0].clear()
    axs[0].plot(x, y_true, label='True', color='black')
    axs[0].plot(x, y_pred, '--', label=f'Pred', color='red')
    axs[0].set_title(f"Signal Prediction at Epoch {epoch}")
    axs[0].set_xlabel("x")
    axs[0].set_ylabel("y")
    axs[0].legend()
    axs[0].grid(True)

    axs[1].clear()
    axs[1].plot(freqs_pos, amp_true, label='True Spectrum', color='black')
    axs[1].plot(freqs_pos, amp_pred, '--', label='Predicted Spectrum', color='red')
    axs[1].set_title(f"Frequency Spectrum at Epoch {epoch}")
    axs[1].set_xlabel("f")
    axs[1].set_ylabel("Amplitude")
    axs[1].set_xlim(0, 6)
    axs[1].set_yscale("log")
    axs[1].grid(True, which="both", ls="--", alpha=0.5)
    axs[1].legend()

anim = FuncAnimation(fig, animate, frames=len(log_epochs), interval=500)

anim.save("mlp_piecewise.gif", writer=PillowWriter(fps=2))
print("Saved animation: mlp_piecewise.gif")


#SOAP

In [14]:
pip install git+https://github.com/haydn-jones/SOAP_JAX

  Cloning https://github.com/haydn-jones/SOAP_JAX to /tmp/pip-req-build-zthgf6vx
  Running command git clone --filter=blob:none --quiet https://github.com/haydn-jones/SOAP_JAX /tmp/pip-req-build-zthgf6vx
  Resolved https://github.com/haydn-jones/SOAP_JAX to commit 2c0c34fa02ae91ead7d8a65c3309a5cdaf8a091c
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 3.2 MB/s eta 0:00:00
  Created wheel for soap-jax: filename=soap_jax-0.1.0-py3-none-any.whl size=5302 sha256=1c8abad54f5360c99a284e2b76f3a4f85672cbb4de485ee0aef6db866292e2fa
  Stored in directory: /tmp/pip-ephem-wheel-cache-_6aipvpl/wheels/53/5f/06/d0c5dd79c713f3c12edd77a51d8cf852437d889024e7d76896
Successfully built soap-jax


In [26]:
import jax
import jax.numpy as jnp
import optax
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

# ============================================================
# 1. Piecewise target function
# ============================================================
def target_function(x):
    left = 5.0 + jnp.sum(jnp.stack([jnp.sin(k * x) for k in range(1, 5)]), axis=0)
    right = jnp.cos(10 * x)
    return jnp.where(x < 0, left, right)

# ============================================================
# 2. Chebyshev recursive basis
# ============================================================
def chebyshev_recursive(x, degree):
    T0 = jnp.ones_like(x)
    T1 = x
    result = [T0, T1]
    for _ in range(2, degree + 1):
        T2 = 2 * x * result[-1] - result[-2]
        result.append(T2)
    return jnp.stack(result, axis=-1)

# ============================================================
# 3. KAN parameter initialization
# ============================================================
def init_params_kan2(layers, degree, key=jax.random.PRNGKey(123)):
    keys = jax.random.split(key, len(layers))
    params = []
    for i in range(len(layers) - 1):
        W = jax.random.normal(
            keys[i],
            (layers[i], layers[i + 1], degree + 1)
        ) / jnp.sqrt(layers[i] * (degree + 1))
        params.append({"W": W})
    return params

# ============================================================
# 4. Forward pass
# ============================================================
def fwd(params, x, activation=jax.nn.tanh):
    X = x.reshape(-1, 1)
    for layer in params:
        W = layer["W"]
        degree = W.shape[-1] - 1
        X = activation(X)
        Xc = chebyshev_recursive(X, degree)
        X = jnp.einsum("nid,iod->no", Xc, W)
    return X[:, 0]

# ============================================================
# 5. Loss
# ============================================================
def mse_loss(params, x, y):
    y_pred = fwd(params, x)
    return jnp.mean((y_pred - y) ** 2)

# ============================================================
# 6. Setup
# ============================================================
layers = [1, 64, 64, 1]
degree = 5
epochs = 40000
log_every = 1000

x = jnp.linspace(-jnp.pi, jnp.pi, 80)
y_true = target_function(x)

params = init_params_kan2(layers, degree)

# ============================================================
# 7. SOAP optimizer
# ============================================================
from soap_jax import soap

optimizer = soap(
    learning_rate=3e-4,
    b1=0.9,
    b2=0.99,
    weight_decay=1e-6,
    precondition_frequency=5,
)

opt_state = optimizer.init(params)

# ============================================================
# 8. Training step
# ============================================================
@jax.jit
def train_step(params, opt_state, x, y):
    loss, grads = jax.value_and_grad(mse_loss)(params, x, y)
    updates, opt_state = optimizer.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss

# ============================================================
# 9. Training loop
# ============================================================
predictions = {}
loss_history = []

epochs_to_plot = [0, 5000, 15000, 25000, 35000, 40000]

for epoch in range(epochs + 1):
    params, opt_state, loss = train_step(params, opt_state, x, y_true)
    loss_history.append(loss)

    if epoch % log_every == 0:
        print(f"[SOAP {epoch:6d}] loss = {loss:.3e}")

    if epoch in epochs_to_plot:
        predictions[epoch] = np.array(fwd(params, x))

# ============================================================
# 10. Animation (signal + spectrum)
# ============================================================
plt.switch_backend("Agg")

dx = float(x[1] - x[0])
freqs = np.fft.fftfreq(len(x), d=dx)
mask = freqs >= 0
freqs_pos = freqs[mask]

fft_true = np.abs(np.fft.fft(np.array(y_true)))[mask]

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
plt.tight_layout()

def animate(i):
    epoch = epochs_to_plot[i]
    y_pred = predictions[epoch]
    fft_pred = np.abs(np.fft.fft(y_pred))[mask]

    axs[0].clear()
    axs[0].plot(x, y_true, "k", label="True")
    axs[0].plot(x, y_pred, "r--", label="Pred")
    axs[0].set_title(f"Epoch {epoch}")
    axs[0].legend()
    axs[0].grid(True)

    axs[1].clear()
    axs[1].plot(freqs_pos, fft_true, "k", label="True")
    axs[1].plot(freqs_pos, fft_pred, "r--", label="Pred")
    axs[1].set_yscale("log")
    axs[1].set_xlim(0, 6)
    axs[1].legend()
    axs[1].grid(True, which="both", ls="--", alpha=0.5)

anim = FuncAnimation(fig, animate, frames=len(epochs_to_plot), interval=800)
anim.save("ckan_disc.gif", writer=PillowWriter(fps=2))
plt.close()

print("✓ Saved animation: ckan_disc.gif")

# ============================================================
# 11. Spectral bias evolution figure (2 × N)
# ============================================================
def fft_mag(signal, dx):
    fft_vals = np.fft.fft(signal)
    freqs = np.fft.fftfreq(len(signal), d=dx)
    mask = freqs >= 0
    return freqs[mask], np.abs(fft_vals[mask])

freqs_true, amp_true = fft_mag(np.array(y_true), dx)

ncols = len(epochs_to_plot)
fig, axes = plt.subplots(2, ncols, figsize=(3.2 * ncols, 6), constrained_layout=True)

for j, ep in enumerate(epochs_to_plot):
    y_pred = predictions[ep]

    # --- Function space ---
    ax = axes[0, j]
    ax.plot(x, y_true, "k", lw=1.8)
    ax.plot(x, y_pred, "r--", lw=1.8)
    ax.set_title(f"Epoch {ep}")
    ax.grid(True, alpha=0.3)
    if j == 0:
        ax.set_ylabel(r"$u(x)$")
    else:
        ax.set_yticklabels([])

    # --- Spectral space ---
    ax = axes[1, j]
    freqs_p, amp_p = fft_mag(y_pred, dx)
    ax.semilogy(freqs_true, amp_true, "k", lw=1.8)
    ax.semilogy(freqs_p, amp_p, "r--", lw=1.8)
    ax.set_xlim(0, 6)
    ax.grid(True, which="both", ls="--", alpha=0.4)
    if j == 0:
        ax.set_ylabel(r"$|\hat{u}(k)|$")
    else:
        ax.set_yticklabels([])
    ax.set_xlabel(r"$k$")

fig.suptitle("Spectral Bias Evolution (KAN + SOAP)", fontsize=16, y=1.05)

plt.savefig("spectral_bias_evolution_kan_piecewise.pdf", dpi=300, bbox_inches="tight")
plt.show()

print("✓ Saved figure: spectral_bias_evolution_kan_piecewise.pdf")


# Final prediction (from your tanh-cKAN code)
y_pred = np.array(fwd(params, x))
y_true_np = np.array(y_true)

rel_l2 = np.linalg.norm(y_pred - y_true_np) / np.linalg.norm(y_true_np)

print("="*60)
print(f"Relative L2 error : {rel_l2:.6e}")
print("="*60)


def spectral_error_metric_1d(u_exact, u_pred, p, L):
    """
    1D version of CMAME spectral error metric

    u_exact, u_pred : arrays of shape (N,)
    p               : 0 (L2), 2 (grad), 4 (laplacian)
    L               : domain length
    """
    N = len(u_exact)

    Ue = np.fft.fft(u_exact)
    Up = np.fft.fft(u_pred)
    E  = Ue - Up

    k = 2 * np.pi * np.fft.fftfreq(N, d=L / N)

    if p == 0:
        weight = np.ones_like(k)
    else:
        weight = np.abs(k)**p

    return np.sum(weight * np.abs(E)**2) / N


L = float(x.max() - x.min())

import numpy as np

E_p0 = spectral_error_metric_1d(y_true_np, y_pred, p=0, L=L)
E_p2 = spectral_error_metric_1d(y_true_np, y_pred, p=2, L=L)
E_p4 = spectral_error_metric_1d(y_true_np, y_pred, p=4, L=L)

print("="*70)
print("CMAME Unified Spectral Errors (1D)")
print(f"p = 0 (L2)        : {E_p0:.6e}   | log10 = {np.log10(E_p0):.3f}")
print(f"p = 2 (Gradient)  : {E_p2:.6e}   | log10 = {np.log10(E_p2):.3f}")
print(f"p = 4 (Laplacian) : {E_p4:.6e}   | log10 = {np.log10(E_p4):.3f}")
print("="*70)


def barron_norm_1d(u, L):
    """
    1D Barron norm approximation via FFT
    """
    N = len(u)
    U_hat = np.fft.fftshift(np.fft.fft(u))
    k = np.fft.fftshift(np.fft.fftfreq(N, d=L / N))
    omega = 2 * np.pi * k
    return np.sum(np.abs(omega) * np.abs(U_hat)) / N

BN_true = barron_norm_1d(y_true_np, L)
BN_pred = barron_norm_1d(y_pred, L)

barron_rel_error = np.abs(BN_pred - BN_true) / BN_true

print("="*60)
print(f"Barron norm (exact) : {BN_true:.6e}")
print(f"Barron norm (pred)  : {BN_pred:.6e}")
print(f"Relative Barron error : {barron_rel_error:.6e}")
print("="*60)


[SOAP      0] loss = 6.321e+00
[SOAP   1000] loss = 1.563e-04
[SOAP   2000] loss = 8.699e-05
[SOAP   3000] loss = 4.454e-05
[SOAP   4000] loss = 1.152e-05
[SOAP   5000] loss = 6.778e-06
[SOAP   6000] loss = 1.488e-06
[SOAP   7000] loss = 1.074e-06
[SOAP   8000] loss = 1.590e-06
[SOAP   9000] loss = 4.978e-07
[SOAP  10000] loss = 3.546e-05
[SOAP  11000] loss = 2.542e-06
[SOAP  12000] loss = 3.547e-07
[SOAP  13000] loss = 4.258e-05
[SOAP  14000] loss = 5.553e-08
[SOAP  15000] loss = 4.601e-05
[SOAP  16000] loss = 7.996e-07
[SOAP  17000] loss = 3.800e-07
[SOAP  18000] loss = 2.896e-05
[SOAP  19000] loss = 2.340e-08
[SOAP  20000] loss = 2.891e-06
[SOAP  21000] loss = 2.697e-06
[SOAP  22000] loss = 2.430e-08
[SOAP  23000] loss = 5.238e-07
[SOAP  24000] loss = 8.936e-07
[SOAP  25000] loss = 1.225e-06
[SOAP  26000] loss = 1.212e-06
[SOAP  27000] loss = 2.469e-06
[SOAP  28000] loss = 2.758e-06
[SOAP  29000] loss = 1.152e-05
[SOAP  30000] loss = 2.190e-07
[SOAP  31000] loss = 2.198e-06
[SOAP  3

In [21]:
import jax
import jax.numpy as jnp
import optax
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

jax.config.update("jax_enable_x64", True)

# ============================================================
# 1. Piecewise target function
# ============================================================
def target_function(x):
    left = 5.0 + jnp.sum(jnp.stack([jnp.sin(k * x) for k in range(1, 5)]), axis=0)
    right = jnp.cos(10.0 * x)
    return jnp.where(x < 0, left, right)

# ============================================================
# 2. Chebyshev recursive basis
# ============================================================
def chebyshev_recursive(x, degree):
    T0 = jnp.ones_like(x)
    T1 = x
    result = [T0, T1]
    for _ in range(2, degree + 1):
        T2 = 2.0 * x * result[-1] - result[-2]
        result.append(T2)
    return jnp.stack(result, axis=-1)  # (N, in_dim, degree+1)

# ============================================================
# 3. tanh-cKAN initialization
# ============================================================
def init_params_tanh_ckan(layers, degree, key=jax.random.PRNGKey(123)):
    keys = jax.random.split(key, len(layers))
    params = []

    # Hidden cKAN layers
    for i in range(len(layers) - 2):
        W = jax.random.normal(
            keys[i],
            (layers[i], layers[i + 1], degree + 1)
        ) / jnp.sqrt(layers[i] * (degree + 1))

        g = jnp.ones((layers[i + 1],))  # channel-wise gate
        params.append({"W": W, "g": g})

    # Final linear layer
    Wf = jax.random.normal(
        keys[-1],
        (layers[-2], layers[-1])
    ) / jnp.sqrt(layers[-2])
    bf = jnp.zeros((layers[-1],))

    params.append({"W": Wf, "b": bf})
    return params

# ============================================================
# 4. Forward pass (tanh-cKAN)
# ============================================================
def fwd(params, x):
    X = x.reshape(-1, 1)
    *hidden, last = params

    for layer in hidden:
        W = layer["W"]
        g = layer["g"]
        degree = W.shape[-1] - 1

        # tanh pre-activation
        X = jnp.tanh(X)

        # Chebyshev lift
        Xc = chebyshev_recursive(X, degree)

        # KAN contraction
        X = jnp.einsum("nid,iod->no", Xc, W)

        # channel-wise gating
        # X = g * X

        # tanh post-activation
        X = jnp.tanh(X)

    return (X @ last["W"] + last["b"])[:, 0]

# ============================================================
# 5. Loss
# ============================================================
def mse_loss(params, x, y):
    y_pred = fwd(params, x)
    return jnp.mean((y_pred - y) ** 2)

# ============================================================
# 6. Setup
# ============================================================
layers = [1, 64, 64, 1]
degree = 5
epochs = 40000
log_every = 1000

epochs_to_plot = [0, 5000, 15000, 25000, 35000, 40000]

x = jnp.linspace(-jnp.pi, jnp.pi, 80)
y_true = target_function(x)

params = init_params_tanh_ckan(layers, degree)

# ============================================================
# 7. SOAP optimizer (exactly your style)
# ============================================================
from soap_jax import soap

optimizer = soap(
    learning_rate=3e-4,
    b1=0.9,
    b2=0.99,
    weight_decay=1e-6,
    precondition_frequency=5,
)

opt_state = optimizer.init(params)

# ============================================================
# 8. Training step
# ============================================================
@jax.jit
def train_step(params, opt_state, x, y):
    loss, grads = jax.value_and_grad(mse_loss)(params, x, y)
    updates, opt_state = optimizer.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss

# ============================================================
# 9. Training loop
# ============================================================
predictions = {}
loss_history = []

for epoch in range(epochs + 1):
    params, opt_state, loss = train_step(params, opt_state, x, y_true)
    loss_history.append(loss)

    if epoch % log_every == 0:
        print(f"[SOAP {epoch:6d}] loss = {loss:.3e}")

    if epoch in epochs_to_plot:
        predictions[epoch] = np.array(fwd(params, x))

# ============================================================
# 10. Animation (signal + spectrum)
# ============================================================
plt.switch_backend("Agg")

dx = float(x[1] - x[0])
freqs = np.fft.fftfreq(len(x), d=dx)
mask = freqs >= 0
freqs_pos = freqs[mask]

fft_true = np.abs(np.fft.fft(np.array(y_true)))[mask]

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
plt.tight_layout()

def animate(i):
    epoch = epochs_to_plot[i]
    y_pred = predictions[epoch]
    fft_pred = np.abs(np.fft.fft(y_pred))[mask]

    axs[0].clear()
    axs[0].plot(x, y_true, "k", label="True")
    axs[0].plot(x, y_pred, "r--", label="Pred")
    axs[0].set_title(f"Epoch {epoch}")
    axs[0].legend()
    axs[0].grid(True)

    axs[1].clear()
    axs[1].plot(freqs_pos, fft_true, "k", label="True")
    axs[1].plot(freqs_pos, fft_pred, "r--", label="Pred")
    axs[1].set_yscale("log")
    axs[1].set_xlim(0, 6)
    axs[1].legend()
    axs[1].grid(True, which="both", ls="--", alpha=0.5)

anim = FuncAnimation(fig, animate, frames=len(epochs_to_plot), interval=800)
anim.save("tanh_ckan_soap.gif", writer=PillowWriter(fps=2))
plt.close()

print("✓ Saved animation: tanh_ckan_soap.gif")

# ============================================================
# 11. Spectral bias evolution figure (2 × N)
# ============================================================
def fft_mag(signal, dx):
    fft_vals = np.fft.fft(signal)
    freqs = np.fft.fftfreq(len(signal), d=dx)
    mask = freqs >= 0
    return freqs[mask], np.abs(fft_vals[mask])

freqs_true, amp_true = fft_mag(np.array(y_true), dx)

ncols = len(epochs_to_plot)
fig, axes = plt.subplots(
    2, ncols,
    figsize=(3.2 * ncols, 6),
    constrained_layout=True
)

for j, ep in enumerate(epochs_to_plot):
    y_pred = predictions[ep]

    # --- Function space ---
    ax = axes[0, j]
    ax.plot(x, y_true, "k", lw=1.8)
    ax.plot(x, y_pred, "r--", lw=1.8)
    ax.set_title(f"Epoch {ep}")
    ax.grid(True, alpha=0.3)
    if j == 0:
        ax.set_ylabel(r"$u(x)$")
    else:
        ax.set_yticklabels([])

    # --- Spectral space ---
    ax = axes[1, j]
    freqs_p, amp_p = fft_mag(y_pred, dx)
    ax.semilogy(freqs_true, amp_true, "k", lw=1.8)
    ax.semilogy(freqs_p, amp_p, "r--", lw=1.8)
    ax.set_xlim(0, 6)
    ax.grid(True, which="both", ls="--", alpha=0.4)
    if j == 0:
        ax.set_ylabel(r"$|\hat{u}(k)|$")
    else:
        ax.set_yticklabels([])
    ax.set_xlabel(r"$k$")

fig.suptitle(
    "Spectral Bias Evolution (tanh-cKAN + SOAP)",
    fontsize=16,
    y=1.05
)

plt.savefig(
    "spectral_bias_evolution_tanh_ckan_soap.pdf",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

print("✓ Saved figure: spectral_bias_evolution_tanh_ckan_soap.pdf")


[SOAP      0] loss = 9.105e+00
[SOAP   1000] loss = 2.945e-02
[SOAP   2000] loss = 2.200e-02
[SOAP   3000] loss = 1.514e-02
[SOAP   4000] loss = 6.704e-03
[SOAP   5000] loss = 7.473e-04
[SOAP   6000] loss = 3.239e-05
[SOAP   7000] loss = 1.280e-06
[SOAP   8000] loss = 5.756e-08
[SOAP   9000] loss = 5.954e-08
[SOAP  10000] loss = 2.757e-07
[SOAP  11000] loss = 2.161e-07
[SOAP  12000] loss = 6.630e-08
[SOAP  13000] loss = 6.793e-08
[SOAP  14000] loss = 5.747e-08
[SOAP  15000] loss = 3.969e-07
[SOAP  16000] loss = 3.324e-08
[SOAP  17000] loss = 1.599e-06
[SOAP  18000] loss = 1.134e-07
[SOAP  19000] loss = 9.080e-07
[SOAP  20000] loss = 1.803e-07
[SOAP  21000] loss = 2.272e-07
[SOAP  22000] loss = 9.379e-09
[SOAP  23000] loss = 2.359e-08
[SOAP  24000] loss = 2.217e-08
[SOAP  25000] loss = 5.274e-08
[SOAP  26000] loss = 7.706e-09
[SOAP  27000] loss = 2.337e-09
[SOAP  28000] loss = 2.953e-06
[SOAP  29000] loss = 1.315e-06
[SOAP  30000] loss = 3.887e-09
[SOAP  31000] loss = 4.333e-09
[SOAP  3

In [25]:
# Final prediction (from your tanh-cKAN code)
y_pred = np.array(fwd(params, x))
y_true_np = np.array(y_true)

rel_l2 = np.linalg.norm(y_pred - y_true_np) / np.linalg.norm(y_true_np)

print("="*60)
print(f"Relative L2 error : {rel_l2:.6e}")
print("="*60)


def spectral_error_metric_1d(u_exact, u_pred, p, L):
    """
    1D version of CMAME spectral error metric

    u_exact, u_pred : arrays of shape (N,)
    p               : 0 (L2), 2 (grad), 4 (laplacian)
    L               : domain length
    """
    N = len(u_exact)

    Ue = np.fft.fft(u_exact)
    Up = np.fft.fft(u_pred)
    E  = Ue - Up

    k = 2 * np.pi * np.fft.fftfreq(N, d=L / N)

    if p == 0:
        weight = np.ones_like(k)
    else:
        weight = np.abs(k)**p

    return np.sum(weight * np.abs(E)**2) / N


L = float(x.max() - x.min())

import numpy as np

E_p0 = spectral_error_metric_1d(y_true_np, y_pred, p=0, L=L)
E_p2 = spectral_error_metric_1d(y_true_np, y_pred, p=2, L=L)
E_p4 = spectral_error_metric_1d(y_true_np, y_pred, p=4, L=L)

print("="*70)
print("CMAME Unified Spectral Errors (1D)")
print(f"p = 0 (L2)        : {E_p0:.6e}   | log10 = {np.log10(E_p0):.3f}")
print(f"p = 2 (Gradient)  : {E_p2:.6e}   | log10 = {np.log10(E_p2):.3f}")
print(f"p = 4 (Laplacian) : {E_p4:.6e}   | log10 = {np.log10(E_p4):.3f}")
print("="*70)


def barron_norm_1d(u, L):
    """
    1D Barron norm approximation via FFT
    """
    N = len(u)
    U_hat = np.fft.fftshift(np.fft.fft(u))
    k = np.fft.fftshift(np.fft.fftfreq(N, d=L / N))
    omega = 2 * np.pi * k
    return np.sum(np.abs(omega) * np.abs(U_hat)) / N

BN_true = barron_norm_1d(y_true_np, L)
BN_pred = barron_norm_1d(y_pred, L)

barron_rel_error = np.abs(BN_pred - BN_true) / BN_true

print("="*60)
print(f"Barron norm (exact) : {BN_true:.6e}")
print(f"Barron norm (pred)  : {BN_pred:.6e}")
print(f"Relative Barron error : {barron_rel_error:.6e}")
print("="*60)


Relative L2 error : 3.456674e-05
CMAME Unified Spectral Errors (1D)
p = 0 (L2)        : 9.122739e-07   | log10 = -6.040
p = 2 (Gradient)  : 1.171665e-04   | log10 = -3.931
p = 4 (Laplacian) : 1.314059e-01   | log10 = -0.881
Barron norm (exact) : 7.038021e+01
Barron norm (pred)  : 7.038010e+01
Relative Barron error : 1.578243e-06


In [37]:
import jax
import jax.numpy as jnp
import optax
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

jax.config.update("jax_enable_x64", True)

# ============================================================
# 1. Piecewise target function
# ============================================================
def target_function(x):
    left = 5.0 + jnp.sum(jnp.stack([jnp.sin(k * x) for k in range(1, 5)]), axis=0)
    right = jnp.cos(10.0 * x)
    return jnp.where(x < 0, left, right)

# ============================================================
# 2. Chebyshev recursive basis
# ============================================================
def chebyshev_recursive(x, degree):
    T0 = jnp.ones_like(x)
    T1 = x
    result = [T0, T1]
    for _ in range(2, degree + 1):
        T2 = 2.0 * x * result[-1] - result[-2]
        result.append(T2)
    return jnp.stack(result, axis=-1)  # (N, in_dim, degree+1)

# ============================================================
# 3. tanh-cKAN initialization
# ============================================================
def init_params_tanh_ckan(layers, degree, key=jax.random.PRNGKey(123)):
    keys = jax.random.split(key, len(layers))
    params = []

    # Hidden cKAN layers
    for i in range(len(layers) - 2):
        W = jax.random.normal(
            keys[i],
            (layers[i], layers[i + 1], degree + 1)
        ) / jnp.sqrt(layers[i] * (degree + 1))

        g = jnp.ones((layers[i + 1],))  # channel-wise gate
        params.append({"W": W, "g": g})

    # Final linear layer
    Wf = jax.random.normal(
        keys[-1],
        (layers[-2], layers[-1])
    ) / jnp.sqrt(layers[-2])
    bf = jnp.zeros((layers[-1],))

    params.append({"W": Wf, "b": bf})
    return params

# ============================================================
# 4. Forward pass (tanh-cKAN)
# ============================================================
def fwd(params, x):
    X = x.reshape(-1, 1)
    *hidden, last = params

    for layer in hidden:
        W = layer["W"]
        g = layer["g"]
        degree = W.shape[-1] - 1

        # tanh pre-activation
        # X = jnp.tanh(X)

        # Chebyshev lift
        Xc = chebyshev_recursive(X, degree)

        # KAN contraction
        X = jnp.einsum("nid,iod->no", Xc, W)

        # channel-wise gating
        # X = g * X

        # tanh post-activation
        X = jnp.tanh(X)

    return (X @ last["W"] + last["b"])[:, 0]

# ============================================================
# 5. Loss
# ============================================================
def mse_loss(params, x, y):
    y_pred = fwd(params, x)
    return jnp.mean((y_pred - y) ** 2)

# ============================================================
# 6. Setup
# ============================================================
layers = [1, 64, 64, 1]
degree = 5
epochs = 40000
log_every = 1000

epochs_to_plot = [0, 5000, 15000, 25000, 35000, 40000]

x = jnp.linspace(-jnp.pi, jnp.pi, 80)
y_true = target_function(x)

params = init_params_tanh_ckan(layers, degree)

# ============================================================
# 7. SOAP optimizer (exactly your style)
# ============================================================
from soap_jax import soap

optimizer = soap(
    learning_rate=3e-4,
    b1=0.9,
    b2=0.99,
    weight_decay=1e-6,
    precondition_frequency=5,
)

opt_state = optimizer.init(params)

# ============================================================
# 8. Training step
# ============================================================
@jax.jit
def train_step(params, opt_state, x, y):
    loss, grads = jax.value_and_grad(mse_loss)(params, x, y)
    updates, opt_state = optimizer.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss

# ============================================================
# 9. Training loop
# ============================================================
predictions = {}
loss_history = []

for epoch in range(epochs + 1):
    params, opt_state, loss = train_step(params, opt_state, x, y_true)
    loss_history.append(loss)

    if epoch % log_every == 0:
        print(f"[SOAP {epoch:6d}] loss = {loss:.3e}")

    if epoch in epochs_to_plot:
        predictions[epoch] = np.array(fwd(params, x))

# ============================================================
# 10. Animation (signal + spectrum)
# ============================================================
plt.switch_backend("Agg")

dx = float(x[1] - x[0])
freqs = np.fft.fftfreq(len(x), d=dx)
mask = freqs >= 0
freqs_pos = freqs[mask]

fft_true = np.abs(np.fft.fft(np.array(y_true)))[mask]

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
plt.tight_layout()

def animate(i):
    epoch = epochs_to_plot[i]
    y_pred = predictions[epoch]
    fft_pred = np.abs(np.fft.fft(y_pred))[mask]

    axs[0].clear()
    axs[0].plot(x, y_true, "k", label="True")
    axs[0].plot(x, y_pred, "r--", label="Pred")
    axs[0].set_title(f"Epoch {epoch}")
    axs[0].legend()
    axs[0].grid(True)

    axs[1].clear()
    axs[1].plot(freqs_pos, fft_true, "k", label="True")
    axs[1].plot(freqs_pos, fft_pred, "r--", label="Pred")
    axs[1].set_yscale("log")
    axs[1].set_xlim(0, 6)
    axs[1].legend()
    axs[1].grid(True, which="both", ls="--", alpha=0.5)

anim = FuncAnimation(fig, animate, frames=len(epochs_to_plot), interval=800)
anim.save("gkan_ckan_soap.gif", writer=PillowWriter(fps=2))
plt.close()

print("✓ Saved animation: tanh_ckan_soap.gif")

# ============================================================
# 11. Spectral bias evolution figure (2 × N)
# ============================================================
def fft_mag(signal, dx):
    fft_vals = np.fft.fft(signal)
    freqs = np.fft.fftfreq(len(signal), d=dx)
    mask = freqs >= 0
    return freqs[mask], np.abs(fft_vals[mask])

freqs_true, amp_true = fft_mag(np.array(y_true), dx)

ncols = len(epochs_to_plot)
fig, axes = plt.subplots(
    2, ncols,
    figsize=(3.2 * ncols, 6),
    constrained_layout=True
)

for j, ep in enumerate(epochs_to_plot):
    y_pred = predictions[ep]

    # --- Function space ---
    ax = axes[0, j]
    ax.plot(x, y_true, "k", lw=1.8)
    ax.plot(x, y_pred, "r--", lw=1.8)
    ax.set_title(f"Epoch {ep}")
    ax.grid(True, alpha=0.3)
    if j == 0:
        ax.set_ylabel(r"$u(x)$")
    else:
        ax.set_yticklabels([])

    # --- Spectral space ---
    ax = axes[1, j]
    freqs_p, amp_p = fft_mag(y_pred, dx)
    ax.semilogy(freqs_true, amp_true, "k", lw=1.8)
    ax.semilogy(freqs_p, amp_p, "r--", lw=1.8)
    ax.set_xlim(0, 6)
    ax.grid(True, which="both", ls="--", alpha=0.4)
    if j == 0:
        ax.set_ylabel(r"$|\hat{u}(k)|$")
    else:
        ax.set_yticklabels([])
    ax.set_xlabel(r"$k$")

fig.suptitle(
    "Spectral Bias Evolution (tanh-cKAN + SOAP)",
    fontsize=16,
    y=1.05
)

plt.savefig(
    "spectral_bias_evolution_gkan_ckan_soap.pdf",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

print("✓ Saved figure: spectral_bias_evolution_tanh_ckan_soap.pdf")

# Final prediction (from your tanh-cKAN code)
y_pred = np.array(fwd(params, x))
y_true_np = np.array(y_true)

rel_l2 = np.linalg.norm(y_pred - y_true_np) / np.linalg.norm(y_true_np)

print("="*60)
print(f"Relative L2 error : {rel_l2:.6e}")
print("="*60)


def spectral_error_metric_1d(u_exact, u_pred, p, L):
    """
    1D version of CMAME spectral error metric

    u_exact, u_pred : arrays of shape (N,)
    p               : 0 (L2), 2 (grad), 4 (laplacian)
    L               : domain length
    """
    N = len(u_exact)

    Ue = np.fft.fft(u_exact)
    Up = np.fft.fft(u_pred)
    E  = Ue - Up

    k = 2 * np.pi * np.fft.fftfreq(N, d=L / N)

    if p == 0:
        weight = np.ones_like(k)
    else:
        weight = np.abs(k)**p

    return np.sum(weight * np.abs(E)**2) / N


L = float(x.max() - x.min())

import numpy as np

E_p0 = spectral_error_metric_1d(y_true_np, y_pred, p=0, L=L)
E_p2 = spectral_error_metric_1d(y_true_np, y_pred, p=2, L=L)
E_p4 = spectral_error_metric_1d(y_true_np, y_pred, p=4, L=L)

print("="*70)
print("CMAME Unified Spectral Errors (1D)")
print(f"p = 0 (L2)        : {E_p0:.6e}   | log10 = {np.log10(E_p0):.3f}")
print(f"p = 2 (Gradient)  : {E_p2:.6e}   | log10 = {np.log10(E_p2):.3f}")
print(f"p = 4 (Laplacian) : {E_p4:.6e}   | log10 = {np.log10(E_p4):.3f}")
print("="*70)


def barron_norm_1d(u, L):
    """
    1D Barron norm approximation via FFT
    """
    N = len(u)
    U_hat = np.fft.fftshift(np.fft.fft(u))
    k = np.fft.fftshift(np.fft.fftfreq(N, d=L / N))
    omega = 2 * np.pi * k
    return np.sum(np.abs(omega) * np.abs(U_hat)) / N

BN_true = barron_norm_1d(y_true_np, L)
BN_pred = barron_norm_1d(y_pred, L)

barron_rel_error = np.abs(BN_pred - BN_true) / BN_true

print("="*60)
print(f"Barron norm (exact) : {BN_true:.6e}")
print(f"Barron norm (pred)  : {BN_pred:.6e}")
print(f"Relative Barron error : {barron_rel_error:.6e}")
print("="*60)



[SOAP      0] loss = 9.099e+00
[SOAP   1000] loss = 4.411e-02
[SOAP   2000] loss = 4.377e-02
[SOAP   3000] loss = 2.616e-02
[SOAP   4000] loss = 2.421e-02
[SOAP   5000] loss = 2.400e-02
[SOAP   6000] loss = 2.393e-02
[SOAP   7000] loss = 8.568e-03
[SOAP   8000] loss = 2.745e-03
[SOAP   9000] loss = 2.294e-03
[SOAP  10000] loss = 2.303e-03
[SOAP  11000] loss = 2.292e-03
[SOAP  12000] loss = 2.749e-03
[SOAP  13000] loss = 2.740e-03
[SOAP  14000] loss = 2.282e-03
[SOAP  15000] loss = 1.730e-03
[SOAP  16000] loss = 1.730e-03
[SOAP  17000] loss = 1.727e-03
[SOAP  18000] loss = 1.167e-03
[SOAP  19000] loss = 1.167e-03
[SOAP  20000] loss = 6.488e-04
[SOAP  21000] loss = 5.094e-04
[SOAP  22000] loss = 3.103e-05
[SOAP  23000] loss = 2.122e-07
[SOAP  24000] loss = 1.969e-12
[SOAP  25000] loss = 2.777e-08
[SOAP  26000] loss = 7.641e-11
[SOAP  27000] loss = 5.222e-10
[SOAP  28000] loss = 1.273e-15
[SOAP  29000] loss = 8.185e-07
[SOAP  30000] loss = 2.831e-09
[SOAP  31000] loss = 1.681e-05
[SOAP  3

#siren

In [31]:
import jax
import jax.numpy as jnp
import optax
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# 1. Piecewise target function (discontinuous derivatives)
# ============================================================
def target_function(x):
    left = 5.0 + jnp.sum(jnp.stack([jnp.sin(k * x) for k in range(1, 5)]), axis=0)
    right = jnp.cos(10 * x)
    return jnp.where(x < 0, left, right)

# ============================================================
# 2. SIREN network
# ============================================================
def siren_layer_init(key, in_dim, out_dim, w0, is_first):
    if is_first:
        bound = 1.0 / in_dim
    else:
        bound = jnp.sqrt(6.0 / in_dim) / w0
    W = jax.random.uniform(key, (in_dim, out_dim), minval=-bound, maxval=bound)
    b = jnp.zeros((out_dim,))
    return {"W": W, "b": b}


def init_siren(layers, w0=30.0, key=jax.random.PRNGKey(0)):
    params = []
    keys = jax.random.split(key, len(layers))
    for i in range(len(layers) - 1):
        params.append(
            siren_layer_init(
                keys[i],
                layers[i],
                layers[i + 1],
                w0=w0,
                is_first=(i == 0),
            )
        )
    return params


def fwd_siren(params, x, w0=30.0):
    X = x.reshape((-1, 1))
    for layer in params[:-1]:
        X = jnp.sin(w0 * (X @ layer["W"] + layer["b"]))
    last = params[-1]
    return X @ last["W"] + last["b"]

# ============================================================
# 3. Loss
# ============================================================
def mse_loss(params, x, y_true):
    y_pred = fwd_siren(params, x)
    return jnp.mean((y_pred[:, 0] - y_true) ** 2)

# ============================================================
# 4. Setup
# ============================================================
layers = [1, 90, 90, 90, 90, 1]
w0 = 15.0
lr = 1e-4
epochs = 40000

x = jnp.linspace(-jnp.pi, jnp.pi, 80)
y_true = target_function(x)

params = init_siren(layers, w0=w0)
optimizer = optax.adam(lr)
opt_state = optimizer.init(params)

@jax.jit
def train_step(params, opt_state):
    loss, grads = jax.value_and_grad(mse_loss)(params, x, y_true)
    updates, opt_state = optimizer.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss

# ============================================================
# 5. Training loop (store snapshots)
# ============================================================
epochs_to_plot = [0, 5000, 15000, 25000, 35000, 40000]
predictions = {}

for epoch in range(epochs + 1):
    params, opt_state, loss = train_step(params, opt_state)

    if epoch in epochs_to_plot:
        predictions[epoch] = np.array(fwd_siren(params, x)[:, 0])

    if epoch % 5000 == 0:
        print(f"[Adam {epoch:6d}] loss = {loss:.3e}")

# ============================================================
# 6. FFT helper
# ============================================================
def fft_mag(signal, dx):
    fft_vals = np.fft.fft(signal)
    freqs = np.fft.fftfreq(len(signal), d=dx)
    mask = freqs >= 0
    return freqs[mask], np.abs(fft_vals[mask])

# ============================================================
# 7. Spectral Bias Evolution Figure (2 × N)
# ============================================================
dx = float(x[1] - x[0])
freqs_true, amp_true = fft_mag(np.array(y_true), dx)

ncols = len(epochs_to_plot)
fig, axes = plt.subplots(
    2, ncols,
    figsize=(3.2 * ncols, 6),
    constrained_layout=True
)

for j, ep in enumerate(epochs_to_plot):
    y_pred = predictions[ep]

    # --- Top row: function space ---
    ax = axes[0, j]
    ax.plot(x, y_true, color="black", lw=1.8)
    ax.plot(x, y_pred, "r--", lw=1.8)
    ax.set_title(f"Epoch {ep}", fontsize=12)
    ax.grid(True, alpha=0.3)

    if j == 0:
        ax.set_ylabel(r"$u(x)$", fontsize=12)
    else:
        ax.set_yticklabels([])

    # --- Bottom row: spectral space ---
    ax = axes[1, j]
    freqs_pred, amp_pred = fft_mag(y_pred, dx)

    ax.semilogy(freqs_true, amp_true, color="black", lw=1.8)
    ax.semilogy(freqs_pred, amp_pred, "r--", lw=1.8)
    ax.set_xlim(0, 6)
    ax.grid(True, which="both", ls="--", alpha=0.4)

    if j == 0:
        ax.set_ylabel(r"$|\hat{u}(k)|$", fontsize=12)
    else:
        ax.set_yticklabels([])

    ax.set_xlabel(r"$k$", fontsize=11)

# Global title
fig.suptitle(
    "Spectral Bias Evolution (SIREN, Piecewise Function)",
    fontsize=16,
    y=1.03
)

plt.savefig(
    "spectral_bias_evolution_siren_piecewise.pdf",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

print("✓ Saved figure: spectral_bias_evolution_siren_piecewise.png")


[Adam      0] loss = 9.447e+00
[Adam   5000] loss = 2.602e-08
[Adam  10000] loss = 1.409e-04
[Adam  15000] loss = 8.425e-06
[Adam  20000] loss = 1.005e-07
[Adam  25000] loss = 1.663e-27
[Adam  30000] loss = 1.337e-07
[Adam  35000] loss = 2.152e-06
[Adam  40000] loss = 2.548e-05
✓ Saved figure: spectral_bias_evolution_siren_piecewise.png


In [32]:

# Final prediction (from your tanh-cKAN code)
y_pred = np.array(fwd_siren(params, x)[:, 0])
y_true_np = np.array(y_true)

rel_l2 = np.linalg.norm(y_pred - y_true_np) / np.linalg.norm(y_true_np)

print("="*60)
print(f"Relative L2 error : {rel_l2:.6e}")
print("="*60)


def spectral_error_metric_1d(u_exact, u_pred, p, L):
    """
    1D version of CMAME spectral error metric

    u_exact, u_pred : arrays of shape (N,)
    p               : 0 (L2), 2 (grad), 4 (laplacian)
    L               : domain length
    """
    N = len(u_exact)

    Ue = np.fft.fft(u_exact)
    Up = np.fft.fft(u_pred)
    E  = Ue - Up

    k = 2 * np.pi * np.fft.fftfreq(N, d=L / N)

    if p == 0:
        weight = np.ones_like(k)
    else:
        weight = np.abs(k)**p

    return np.sum(weight * np.abs(E)**2) / N


L = float(x.max() - x.min())

import numpy as np

E_p0 = spectral_error_metric_1d(y_true_np, y_pred, p=0, L=L)
E_p2 = spectral_error_metric_1d(y_true_np, y_pred, p=2, L=L)
E_p4 = spectral_error_metric_1d(y_true_np, y_pred, p=4, L=L)

print("="*70)
print("CMAME Unified Spectral Errors (1D)")
print(f"p = 0 (L2)        : {E_p0:.6e}   | log10 = {np.log10(E_p0):.3f}")
print(f"p = 2 (Gradient)  : {E_p2:.6e}   | log10 = {np.log10(E_p2):.3f}")
print(f"p = 4 (Laplacian) : {E_p4:.6e}   | log10 = {np.log10(E_p4):.3f}")
print("="*70)


def barron_norm_1d(u, L):
    """
    1D Barron norm approximation via FFT
    """
    N = len(u)
    U_hat = np.fft.fftshift(np.fft.fft(u))
    k = np.fft.fftshift(np.fft.fftfreq(N, d=L / N))
    omega = 2 * np.pi * k
    return np.sum(np.abs(omega) * np.abs(U_hat)) / N

BN_true = barron_norm_1d(y_true_np, L)
BN_pred = barron_norm_1d(y_pred, L)

barron_rel_error = np.abs(BN_pred - BN_true) / BN_true

print("="*60)
print(f"Barron norm (exact) : {BN_true:.6e}")
print(f"Barron norm (pred)  : {BN_pred:.6e}")
print(f"Relative Barron error : {barron_rel_error:.6e}")
print("="*60)



Relative L2 error : 1.388245e-03
CMAME Unified Spectral Errors (1D)
p = 0 (L2)        : 1.471434e-03   | log10 = -2.832
p = 2 (Gradient)  : 7.946690e-01   | log10 = -0.100
p = 4 (Laplacian) : 7.683145e+02   | log10 = 2.886
Barron norm (exact) : 7.038021e+01
Barron norm (pred)  : 7.066649e+01
Relative Barron error : 4.067590e-03


In [33]:
import jax
import jax.numpy as jnp
import optax
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# 1. Target function (multi-frequency signal)
# ============================================================
def target_function(t):
    return (
        jnp.sin(2 * jnp.pi * 0.01 * t)
        + 0.5 * jnp.sin(2 * jnp.pi * 0.05 * t)
        + 0.2 * jnp.sin(2 * jnp.pi * 0.1 * t)
    )

# ============================================================
# 2. SIREN definition
# ============================================================
def siren_layer_init(key, in_dim, out_dim, w0, is_first):
    if is_first:
        bound = 1.0 / in_dim
    else:
        bound = jnp.sqrt(6.0 / in_dim) / w0
    W = jax.random.uniform(key, (in_dim, out_dim), minval=-bound, maxval=bound)
    b = jnp.zeros((out_dim,))
    return {"W": W, "b": b}


def init_siren(layers, w0=10.0, key=jax.random.PRNGKey(0)):
    params = []
    keys = jax.random.split(key, len(layers))
    for i in range(len(layers) - 1):
        params.append(
            siren_layer_init(
                keys[i],
                layers[i],
                layers[i + 1],
                w0=w0,
                is_first=(i == 0),
            )
        )
    return params


def fwd_siren(params, t, w0=10.0):
    x = t.reshape((-1, 1))
    for layer in params[:-1]:
        x = jnp.sin(w0 * (x @ layer["W"] + layer["b"]))
    last = params[-1]
    return x @ last["W"] + last["b"]

# ============================================================
# 3. Loss
# ============================================================
def mse_loss(params, t, y_true):
    y_pred = fwd_siren(params, t)
    return jnp.mean((y_pred[:, 0] - y_true) ** 2)

# ============================================================
# 4. Setup
# ============================================================
layers = [1, 64, 64, 1]
w0 = 10.0
epochs = 40000
lr = 1e-4

t = jnp.linspace(0, 300, 301)
y_true = target_function(t)

params = init_siren(layers, w0=w0)

optimizer = optax.adam(lr)
opt_state = optimizer.init(params)

@jax.jit
def train_step(params, opt_state):
    loss, grads = jax.value_and_grad(mse_loss)(params, t, y_true)
    updates, opt_state = optimizer.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss

# ============================================================
# 5. Training loop (store snapshots)
# ============================================================
epochs_to_plot = [0, 5000, 15000, 25000, 35000, 40000]
predictions = {}

for epoch in range(epochs + 1):
    params, opt_state, loss = train_step(params, opt_state)

    if epoch in epochs_to_plot:
        predictions[epoch] = np.array(fwd_siren(params, t)[:, 0])

    if epoch % 5000 == 0:
        print(f"[Adam {epoch:6d}] loss = {loss:.3e}")

# ============================================================
# 6. FFT helper
# ============================================================
def fft_mag(signal, dt):
    fft_vals = np.fft.fft(signal)
    freqs = np.fft.fftfreq(len(signal), d=dt)
    mask = freqs >= 0
    return freqs[mask], np.abs(fft_vals[mask])

# ============================================================
# 7. Spectral Learning Evolution Figure (2 × N)
# ============================================================
dt = float(t[1] - t[0])
freqs_true, amp_true = fft_mag(np.array(y_true), dt)

ncols = len(epochs_to_plot)
fig, axes = plt.subplots(
    2, ncols,
    figsize=(3.2 * ncols, 6),
    constrained_layout=True
)

for j, ep in enumerate(epochs_to_plot):
    y_pred = predictions[ep]

    # --- Top row: physical space ---
    ax = axes[0, j]
    ax.plot(t, y_true, color="black", lw=1.8)
    ax.plot(t, y_pred, "r--", lw=1.8)
    ax.set_title(f"Epoch {ep}", fontsize=12)
    ax.grid(True, alpha=0.3)

    if j == 0:
        ax.set_ylabel(r"$u(x)$", fontsize=12)
    else:
        ax.set_yticklabels([])

    # --- Bottom row: Fourier spectrum ---
    ax = axes[1, j]
    freqs_pred, amp_pred = fft_mag(y_pred, dt)

    ax.semilogy(freqs_true, amp_true, color="black", lw=1.8)
    ax.semilogy(freqs_pred, amp_pred, "r--", lw=1.8)
    ax.set_xlim(0, 0.5)
    ax.grid(True, which="both", ls="--", alpha=0.4)

    if j == 0:
        ax.set_ylabel(r"$|\hat{u}(k)|$", fontsize=12)
    else:
        ax.set_yticklabels([])

    ax.set_xlabel(r"$k$", fontsize=11)

# Global title
fig.suptitle(
    "Spectral Learning Pattern Evolution (SIREN)",
    fontsize=16,
    y=1.03
)

plt.savefig(
    "spectral_learning_evolution_siren.pdf",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

print("✓ Saved figure: spectral_learning_evolution_siren.png")




[Adam      0] loss = 6.455e-01
[Adam   5000] loss = 4.054e-07
[Adam  10000] loss = 8.863e-07
[Adam  15000] loss = 8.829e-07
[Adam  20000] loss = 1.052e-06
[Adam  25000] loss = 6.233e-06
[Adam  30000] loss = 6.109e-06
[Adam  35000] loss = 2.269e-06
[Adam  40000] loss = 1.826e-06
✓ Saved figure: spectral_learning_evolution_siren.png


ValueError: operands could not be broadcast together with shapes (80,) (301,) 

In [36]:


# Final prediction (from your tanh-cKAN code)
y_pred = np.array(fwd_siren(params, t, w0)[:, 0])
y_true_np = np.array(y_true)

rel_l2 = np.linalg.norm(y_pred - y_true_np) / np.linalg.norm(y_true_np)

print("="*60)
print(f"Relative L2 error : {rel_l2:.6e}")
print("="*60)


def spectral_error_metric_1d(u_exact, u_pred, p, L):
    """
    1D version of CMAME spectral error metric

    u_exact, u_pred : arrays of shape (N,)
    p               : 0 (L2), 2 (grad), 4 (laplacian)
    L               : domain length
    """
    N = len(u_exact)

    Ue = np.fft.fft(u_exact)
    Up = np.fft.fft(u_pred)
    E  = Ue - Up

    k = 2 * np.pi * np.fft.fftfreq(N, d=L / N)

    if p == 0:
        weight = np.ones_like(k)
    else:
        weight = np.abs(k)**p

    return np.sum(weight * np.abs(E)**2) / N


L = float(x.max() - x.min())

import numpy as np

E_p0 = spectral_error_metric_1d(y_true_np, y_pred, p=0, L=L)
E_p2 = spectral_error_metric_1d(y_true_np, y_pred, p=2, L=L)
E_p4 = spectral_error_metric_1d(y_true_np, y_pred, p=4, L=L)

print("="*70)
print("CMAME Unified Spectral Errors (1D)")
print(f"p = 0 (L2)        : {E_p0:.6e}   | log10 = {np.log10(E_p0):.3f}")
print(f"p = 2 (Gradient)  : {E_p2:.6e}   | log10 = {np.log10(E_p2):.3f}")
print(f"p = 4 (Laplacian) : {E_p4:.6e}   | log10 = {np.log10(E_p4):.3f}")
print("="*70)


def barron_norm_1d(u, L):
    """
    1D Barron norm approximation via FFT
    """
    N = len(u)
    U_hat = np.fft.fftshift(np.fft.fft(u))
    k = np.fft.fftshift(np.fft.fftfreq(N, d=L / N))
    omega = 2 * np.pi * k
    return np.sum(np.abs(omega) * np.abs(U_hat)) / N

BN_true = barron_norm_1d(y_true_np, L)
BN_pred = barron_norm_1d(y_pred, L)

barron_rel_error = np.abs(BN_pred - BN_true) / BN_true

print("="*60)
print(f"Barron norm (exact) : {BN_true:.6e}")
print(f"Barron norm (pred)  : {BN_pred:.6e}")
print(f"Relative Barron error : {barron_rel_error:.6e}")
print("="*60)


Relative L2 error : 1.802846e-03
CMAME Unified Spectral Errors (1D)
p = 0 (L2)        : 6.289243e-04   | log10 = -3.201
p = 2 (Gradient)  : 3.976225e+00   | log10 = 0.599
p = 4 (Laplacian) : 4.913799e+04   | log10 = 4.691
Barron norm (exact) : 3.778953e+01
Barron norm (pred)  : 3.808629e+01
Relative Barron error : 7.852968e-03
